In [ ]:
import os
import math
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.stattools import acf, pacf


warnings.filterwarnings("ignore")

#### Design a Univariate/Multivariate Stochastic Volatility model 

In [ ]:
def _infer_dim_from_params(phi, sigma_eta, sigma_eps, xi, d=None):
    if d is not None:
        if not isinstance(d, int) or d <= 0:
            raise ValueError("d must be a positive integer")
        return d

    candidates = []

    for obj in [phi, sigma_eta, sigma_eps, xi]:
        t = tf.convert_to_tensor(obj)
        if t.shape.rank == 0:
            continue
        if t.shape.rank == 1:
            candidates.append(int(t.shape[0]))
        elif t.shape.rank == 2:
            if t.shape[0] != t.shape[1]:
                raise ValueError("Matrix parameters must be square")
            candidates.append(int(t.shape[0]))
        else:
            raise ValueError("Parameters must be scalar, vector, or matrix")

    if len(candidates) == 0:
        return 1

    d0 = candidates[0]
    if any(c != d0 for c in candidates):
        raise ValueError("Incompatible dimensions across phi / sigma_eta / sigma_eps / xi")
    return d0


def _make_covariance_and_chol_tf(sigma_like, d, dtype, name):
    """
    Accepts:
      - scalar std
      - vector of stds, length d
      - covariance matrix (d,d)
    Returns:
      Q, chol(Q)
    """
    sigma_like = tf.convert_to_tensor(sigma_like, dtype=dtype)

    if sigma_like.shape.rank == 0:
        tf.debugging.assert_positive(sigma_like, message=f"{name} must be positive")
        Q = tf.eye(d, dtype=dtype) * sigma_like**2
        L = tf.eye(d, dtype=dtype) * sigma_like
        return Q, L

    if sigma_like.shape.rank == 1:
        tf.debugging.assert_equal(
            tf.shape(sigma_like)[0], d,
            message=f"{name} vector must have length d"
        )
        tf.debugging.assert_positive(
            sigma_like,
            message=f"all entries of {name} must be positive"
        )
        Q = tf.linalg.diag(sigma_like**2)
        L = tf.linalg.diag(sigma_like)
        return Q, L

    if sigma_like.shape.rank == 2:
        tf.debugging.assert_equal(tf.shape(sigma_like)[0], d,
            message=f"{name} matrix must have shape (d,d)")
        tf.debugging.assert_equal(tf.shape(sigma_like)[1], d,
            message=f"{name} matrix must have shape (d,d)")
        L = tf.linalg.cholesky(sigma_like)
        return sigma_like, L

    raise ValueError(f"{name} must be scalar, vector, or matrix")


def _make_scale_vector_tf(xi, d, dtype):
    """
    Scalar to length-d vector, or validate length-d vector.
    Enforces positivity.
    """
    xi_vec = _as_vector_param_tf(xi, d, dtype, "xi")
    tf.debugging.assert_positive(xi_vec, message="xi must be positive")
    return xi_vec


def _as_vector_param_tf(x, d, dtype, name):
    """
    Convert scalar to vector of length d
    or validate vector of length d.
    """
    x = tf.convert_to_tensor(x, dtype=dtype)

    if x.shape.rank == 0:
        return tf.fill([d], x)

    if x.shape.rank == 1:
        tf.debugging.assert_equal(
            tf.shape(x)[0], d,
            message=f"{name} must have length d"
        )
        return x

    raise ValueError(f"{name} must be scalar or a rank-1 tensor")

def _as_transition_matrix_tf(phi, d, dtype):
    """
    phi can be:
      - scalar: phi * I
      - vector: diag(phi)
      - matrix: full transition matrix
    """
    phi = tf.convert_to_tensor(phi, dtype=dtype)

    if phi.shape.rank == 0:
        tf.debugging.assert_less(
            tf.abs(phi), tf.cast(1.0, dtype),
            message="scalar phi must satisfy |phi| < 1"
        )
        return phi * tf.eye(d, dtype=dtype)

    if phi.shape.rank == 1:
        tf.debugging.assert_equal(
            tf.shape(phi)[0], d,
            message="phi vector must have length d"
        )
        tf.debugging.assert_less(
            tf.reduce_max(tf.abs(phi)),
            tf.cast(1.0, dtype),
            message="all phi entries must satisfy |phi_i| < 1"
        )
        return tf.linalg.diag(phi)

    if phi.shape.rank == 2:
        tf.debugging.assert_equal(
            tf.shape(phi)[0], d,
            message="phi matrix must have shape (d,d)"
        )
        tf.debugging.assert_equal(
            tf.shape(phi)[1], d,
            message="phi matrix must have shape (d,d)"
        )
        eigvals = tf.linalg.eigvals(tf.cast(phi, tf.complex128))
        tf.debugging.assert_less(
            tf.reduce_max(tf.abs(eigvals)),
            tf.constant(1.0, dtype=tf.float64),
            message="phi matrix must be stable: spectral radius < 1"
        )
        return phi

    raise ValueError("phi must be scalar, vector, or matrix")


def _stationary_covariance_ar1(Phi, Q, dtype):
    """
    Solve vec(P) = (I - Phi⊗Phi)^(-1) vec(Q)
    for the stationary covariance of h_t = Phi h_{t-1} + eta_t.
    """
    d = int(Phi.shape[0])
    kron_term = tf.experimental.numpy.kron(Phi, Phi)
    I = tf.eye(d * d, dtype=dtype)
    vecQ = tf.reshape(Q, (-1, 1))
    vecP = tf.linalg.solve(I - kron_term, vecQ)
    P = tf.reshape(vecP, (d, d))
    P = 0.5 * (P + tf.transpose(P))  # symmetrize
    return P



def _as_noise_chol_tf(sigma_like, d, dtype, name):
    """
    Accept:
      - scalar std
      - vector of stds length d
      - covariance matrix (d,d)

    Return Cholesky factor L such that noise = L z, z~N(0,I).
    Pure TensorFlow, graph-safe.
    """
    sigma_like = tf.convert_to_tensor(sigma_like, dtype=dtype)

    if sigma_like.shape.rank == 0:
        tf.debugging.assert_positive(sigma_like, message=f"{name} must be positive")
        return sigma_like * tf.eye(d, dtype=dtype)

    if sigma_like.shape.rank == 1:
        tf.debugging.assert_equal(
            tf.shape(sigma_like)[0], d,
            message=f"{name} vector must have length d"
        )
        tf.debugging.assert_positive(
            sigma_like,
            message=f"all entries of {name} must be positive"
        )
        return tf.linalg.diag(sigma_like)

    if sigma_like.shape.rank == 2:
        tf.debugging.assert_equal(
            tf.shape(sigma_like)[0], d,
            message=f"{name} matrix must have shape (d,d)"
        )
        tf.debugging.assert_equal(
            tf.shape(sigma_like)[1], d,
            message=f"{name} matrix must have shape (d,d)"
        )
        return tf.linalg.cholesky(sigma_like)

    raise ValueError(f"{name} must be scalar, vector, or matrix")



def SV_model_sim_tf_h(iT, phi, sigma_eta, sigma_eps, xi, seed=123, d=None, dtype=tf.float32):
    """
    Gaussian stochastic volatility model simulator

    Supporting:
      - univariate Y shape (T,)
      - multivariate Y shape (T,d) 
    """
    if not isinstance(iT, int) or iT <= 0:
        raise ValueError("iT must be a positive integer")

    if seed is not None:
        tf.random.set_seed(seed)

    d = _infer_dim_from_params(phi, sigma_eta, sigma_eps, xi, d=d)

    Phi = _as_transition_matrix_tf(phi, d, dtype)
    Q_eta, chol_eta = _make_covariance_and_chol_tf(sigma_eta, d, dtype, "sigma_eta")
    _, chol_eps = _make_covariance_and_chol_tf(sigma_eps, d, dtype, "sigma_eps")
    xi_vec = _make_scale_vector_tf(xi, d, dtype)

    z_eta = tf.random.normal((iT, d), dtype=dtype)
    z_eps = tf.random.normal((iT, d), dtype=dtype)

    eta = tf.linalg.matmul(z_eta, tf.transpose(chol_eta))
    eps = tf.linalg.matmul(z_eps, tf.transpose(chol_eps))

    P0 = _stationary_covariance_ar1(Phi, Q_eta, dtype)
    chol_P0 = tf.linalg.cholesky(P0 + tf.cast(1e-12, dtype) * tf.eye(d, dtype=dtype))

    z0 = tf.random.normal((d,), dtype=dtype)
    h0 = tf.linalg.matvec(chol_P0, z0)

    if iT == 1:
        h = h0[None, :]
    else:
        h_rest = tf.scan(
            lambda h_prev, eta_t: tf.linalg.matvec(Phi, h_prev) + eta_t,
            eta[1:],
            initializer=h0
        )
        h = tf.concat([h0[None, :], h_rest], axis=0)

    y = xi_vec[None, :] * eps * tf.exp(h / tf.cast(2.0, dtype))

    if d == 1:
        y = tf.squeeze(y, axis=-1)
        h = tf.squeeze(h, axis=-1)

    tf.debugging.assert_all_finite(h, "the state contains NaN/Inf")
    tf.debugging.assert_all_finite(y, "observations contain NaN/Inf")

    return {"vY": y, "h": h}

#### Implementation of a Bootstrap Particle Filter (BPF)
BPF utilities in addition to those of the SV model above

In [ ]:
def make_prop_sv(phi, sigma_eta, dtype=tf.float32):
    """
    SV state propagation:
        h_t = Phi h_{t-1} + eta_t

    Works for:
      - univariate: phi scalar, sigma_eta scalar
      - multivariate independent: phi scalar/vector, sigma_eta scalar/vector
      - multivariate correlated state noise: sigma_eta covariance matrix

    Input x must be shape (d,).
    """
    phi = tf.convert_to_tensor(phi, dtype=dtype)
    sigma_eta = tf.convert_to_tensor(sigma_eta, dtype=dtype)

    @tf.function(reduce_retracing=True)
    def prop_sv(x):
        x = tf.convert_to_tensor(x, dtype=dtype)

        tf.debugging.assert_rank(x, 1, message="state x must have shape (d,)")

        d = tf.shape(x)[0]

        Phi = _as_transition_matrix_tf(phi, d, dtype)
        L_eta = _as_noise_chol_tf(sigma_eta, d, dtype, "sigma_eta")

        z = tf.random.normal(tf.shape(x), dtype=dtype)
        eta = tf.linalg.matvec(L_eta, z)

        x_next = tf.linalg.matvec(Phi, x) + eta
        tf.debugging.assert_all_finite(x_next, "prop_sv produced NaN/Inf")

        return x_next

    return prop_sv


def make_loglik_sv(sigma_eps, xi, dtype=tf.float32):
    """
    Gaussian SV observation log-likelihood under conditional independence:
        y_t = xi * exp(h_t / 2) * eps_t

    particles: (Np, d)  
    y:         (d,)
    returns:   (Np,)
    """
    sigma_eps = tf.convert_to_tensor(sigma_eps, dtype=dtype)
    xi = tf.convert_to_tensor(xi, dtype=dtype)

    @tf.function(reduce_retracing=True)
    def loglik_sv(particles, y):
        particles = tf.convert_to_tensor(particles, dtype=dtype)
        y = tf.convert_to_tensor(y, dtype=dtype)

        tf.debugging.assert_rank(particles, 2, message="particles must have shape (Np, d)")
        tf.debugging.assert_rank(y, 1, message="y must have shape (d,)")

        d = tf.shape(particles)[1]
        tf.debugging.assert_equal(
            tf.shape(y)[0], d,
            message="y dimension must match particle dimension"
        )

        sigma_eps_vec = _as_vector_param_tf(sigma_eps, d, dtype, "sigma_eps")
        xi_vec = _as_vector_param_tf(xi, d, dtype, "xi")

        tf.debugging.assert_positive(sigma_eps_vec, message="sigma_eps must be positive")
        tf.debugging.assert_positive(xi_vec, message="xi must be positive")

        # log variance for each particle and dimension:
        # log var = log(xi^2 sigma_eps^2 exp(h)) = 2 log xi + 2 log sigma_eps + h
        log_var = (
            2.0 * tf.math.log(xi_vec)[None, :]
            + 2.0 * tf.math.log(sigma_eps_vec)[None, :]
            + particles
        )

        var = tf.exp(log_var)
        log_2pi = log_2pi = tf.math.log(tf.constant(2.0 * math.pi, dtype=dtype)) 

        llk = -0.5 * tf.reduce_sum(
            log_2pi + log_var + tf.square(y[None, :]) / var,
            axis=1
        )

        tf.debugging.assert_all_finite(llk, "SV log-likelihood produced NaN/Inf")
        return llk

    return loglik_sv


def multinomial_resampling(particles, weights): 

    particles = tf.convert_to_tensor(particles)
    dtype = particles.dtype
    weights = tf.convert_to_tensor(weights, dtype=dtype)
    
    N = tf.shape(particles)[0]
    idx = tf.random.categorical(tf.expand_dims(tf.math.log(weights), 0), N)[0]
    new_particles = tf.gather(particles, idx)
    new_weights = tf.ones_like(weights)/tf.cast(N, dtype=dtype)

    return new_particles, new_weights


def bpf_generic_resampling(
    Y,
    Np,
    prop_fn,
    log_likelihood_fn,
    resampling_fn=multinomial_resampling,
    resample_threshold=False,
    dtype=tf.float32,
    carry_resampled_weights=False,
    eps=1e-12,
):
    """
    Generic bootstrap particle filter.

    carry_resampled_weights=False:
        use for multinomial / OT / equal-weight output resamplers

    carry_resampled_weights=True:
        use for no-resampling / PFNet soft-resampling
    """
    if not callable(prop_fn):
        raise TypeError("prop_fn must be callable")
    if not callable(log_likelihood_fn):
        raise TypeError("log_likelihood_fn must be callable")
    if not callable(resampling_fn):
        raise TypeError("resampling_fn must be callable")
    if not isinstance(Np, int) or Np <= 1:
        raise ValueError("Np must be an integer > 1")

    Y = tf.cast(tf.convert_to_tensor(Y), dtype)
    if len(Y.shape) != 2:
        raise ValueError("Y must have shape (T, d)")
    if Y.shape[0] < 1:
        raise ValueError("Y must contain at least one time step")

    T, d = Y.shape
    eps = tf.cast(eps, dtype)

    particles = tf.zeros([Np, d], dtype=dtype)
    log_weights = -tf.math.log(tf.cast(Np, dtype)) * tf.ones([Np], dtype=dtype)

    total_loglik = tf.constant(0.0, dtype=dtype)
    ests, ESSs = [], []

    threshold_value = tf.cast(Np / 2.0, dtype)

    for t in range(T):
        y_t = Y[t]

        # Propagation
        particles = tf.vectorized_map(prop_fn, particles)

        # likelihood
        llk = tf.cast(log_likelihood_fn(particles, y_t), dtype)
        tf.debugging.assert_all_finite(llk, "llk has NaN/Inf")

        # generic log-weight recursion
        log_w_unnorm = log_weights + llk
        loglik_t = tf.reduce_logsumexp(log_w_unnorm)
        total_loglik += loglik_t

        # normalize in log-space
        log_weights = log_w_unnorm - loglik_t
        weights = tf.exp(log_weights)

        tf.debugging.assert_all_finite(weights, "weights have NaN/Inf")

        # estimate
        est = tf.reduce_sum(particles * weights[:, None], axis=0)
        ests.append(est)

        # ESS
        ESS = 1.0 / tf.reduce_sum(tf.square(weights))
        ESSs.append(ESS)

        do_resample = ESS < threshold_value if resample_threshold else True

        if do_resample:
            particles, new_weights = resampling_fn(particles, weights)
            new_weights = tf.cast(new_weights, dtype)
            tf.debugging.assert_all_finite(new_weights, "resampled weights have NaN/Inf")

            if carry_resampled_weights:
                new_weights = tf.maximum(new_weights, eps)
                new_weights = new_weights / tf.reduce_sum(new_weights)
                log_weights = tf.math.log(new_weights)
            else:
                # reset to uniform after equal-weight resamplers
                log_weights = -tf.math.log(tf.cast(Np, dtype)) * tf.ones([Np], dtype=dtype)

    return tf.stack(ests), tf.stack(ESSs), total_loglik

#### Implementation of the Extended and Unscented Kalman Filter (EKF/UKF)

In [ ]:
#### EKF/UKF CORE 
@tf.function(reduce_retracing=True)
def unscented_sigma_points_batch(x_particles, P, alpha, beta, kappa): 
    """
    x_particles: (Np, d) particle states
    P: (d, d) covariance matrix
    Returns:
        sigma_points: (Np, 2d+1, d)
        Wm: (2d+1,)
        Wc: (2d+1,)
    """
    dtype = x_particles.dtype
    P = tf.cast(P, dtype)

    shape = tf.shape(x_particles)
    Np = shape[0]
    d = shape[1]
    d_float = tf.cast(d, dtype)

    lam = alpha**2 * (d_float + kappa) - d_float
    c = d_float + lam

    # Weights
    Wm = tf.concat([[lam / c], tf.fill([2*d], 1/(2*c))], axis=0)
    Wc = tf.concat([[lam / c + (1 - alpha**2 + beta)], tf.fill([2*d], 1/(2*c))], axis=0)

    # Cholesky scaled
    sqrtP = tf.linalg.cholesky(c * P)  # (d,d)
    
    # Reshape
    sqrtP = tf.reshape(sqrtP, (1, d, d))  # (1,d,d)
    x_exp = tf.expand_dims(x_particles, 1) # (Np,1,d)
    
    sigma_plus = x_exp + tf.transpose(sqrtP, perm=[0,2,1])  # (Np,d,d)
    sigma_minus = x_exp - tf.transpose(sqrtP, perm=[0,2,1])

    sigma_points = tf.concat([x_exp, sigma_plus, sigma_minus], axis=1)  # (Np, 2d+1, d)

    return sigma_points, Wm, Wc


def unscented_sigma_points(x, P, alpha, beta, kappa):
    sigma, Wm, Wc = unscented_sigma_points_batch(x[None, :], P[None, :, :],alpha, beta, kappa) 
    
    return sigma[0], Wm, Wc


def unscented_transform(sigma, Wm, Wc, noise_cov=None):
    
    mean = tf.reduce_sum(sigma * Wm[:, None], axis=0)
    diff = sigma - mean
    cov = tf.einsum('i,ij,ik->jk', Wc, diff, diff) 

    if noise_cov is not None:
        cov = cov + noise_cov

    return mean, cov, diff



@tf.function(reduce_retracing=True)
def ukf_predict(x, P, f, Q, alpha, beta, kappa, t):

    # -------------------------
    # Input validation 
    # -------------------------
    tf.debugging.assert_all_finite(x, "x contains NaN or Inf")
    tf.debugging.assert_all_finite(P, "P contains NaN or Inf")
    tf.debugging.assert_all_finite(Q, "Q contains NaN or Inf")

    tf.debugging.assert_equal(tf.shape(x)[0], tf.shape(P)[0], message="x and P dimension mismatch")
    tf.debugging.assert_equal(tf.shape(P)[0], tf.shape(P)[1], message="P must be square")

    # Symmetry check 
    tf.debugging.assert_near(P, tf.transpose(P), atol=1e-8, message="P is not symmetric")

    
    dtype = x.dtype
    sigma, Wm, Wc = unscented_sigma_points(x, P, alpha, beta, kappa)
    
    # Transform sigma points 
    sigma_f = tf.vectorized_map(lambda s: f(s, t), sigma)
    sigma_f = tf.cast(sigma_f, dtype)
    
    x_pred, P_pred, _ = unscented_transform(sigma_f, Wm, Wc, Q)
    return x_pred, P_pred


@tf.function(reduce_retracing=True)
def ukf_update_check(x_pred, P_pred, y, h, R, alpha=1e-3, beta=2.0, kappa=0.0, t=0):

    # --------------------------------------------------------
    # Input validation 
    # --------------------------------------------------------
    if not tf.is_tensor(x_pred):
        raise TypeError("x_pred must be a TensorFlow tensor")

    if not tf.is_tensor(P_pred):
        raise TypeError("P_pred must be a TensorFlow tensor")

    if not tf.is_tensor(y):
        raise TypeError("y must be a TensorFlow tensor")

    if not tf.is_tensor(R):
        raise TypeError("R must be a TensorFlow tensor")

    if not callable(h):
        raise TypeError("h must be a callable function")

    if len(x_pred.shape) != 1:
        raise ValueError("x_pred must be a 1D tensor")

    if len(P_pred.shape) != 2:
        raise ValueError("P_pred must be a 2D tensor")

    if P_pred.shape[0] != P_pred.shape[1]:
        raise ValueError("P_pred must be a square matrix")

    if x_pred.shape[0] != P_pred.shape[0]:
        raise ValueError("x_pred and P_pred dimension mismatch")

    if y.shape[0] < 1:
        raise ValueError("y must be a non-empty vector")

    # --------------------------------------------------------
    # Cast 
    # --------------------------------------------------------
    dtype = x_pred.dtype
    P_pred = tf.cast(P_pred, dtype)
    y = tf.cast(y, dtype)
    R = tf.cast(R, dtype)

    n = tf.shape(x_pred)[0]

    # --------------------------------------------------------
    # Checks
    # --------------------------------------------------------
    tf.debugging.assert_all_finite(x_pred, "x_pred contains NaNs or Infs")
    tf.debugging.assert_all_finite(P_pred, "P_pred contains NaNs or Infs")
    tf.debugging.assert_all_finite(y, "y contains NaNs or Infs")
    tf.debugging.assert_all_finite(R, "R contains NaNs or Infs")

    tf.debugging.assert_near(
        P_pred, tf.transpose(P_pred), atol=1e-8,
        message="P_pred is not symmetric"
    )

    # PD check
    sqrt_test = tf.linalg.cholesky(P_pred)  

    # --------------------------------------------------------
    # UKF parameters
    # --------------------------------------------------------
    lam = alpha**2 * (tf.cast(n, dtype) + kappa) - tf.cast(n, dtype)
    c = tf.cast(n, dtype) + lam

    Wm = tf.concat([[lam / c], tf.fill([2*n], 1/(2*c))], axis=0)
    Wc = tf.concat([[lam / c + (1 - alpha**2 + beta)], tf.fill([2*n], 1/(2*c))], axis=0)

    Wm = tf.cast(Wm, dtype)
    Wc = tf.cast(Wc, dtype)

    # --------------------------------------------------------
    # Sigma points
    # --------------------------------------------------------
    sqrtP = tf.linalg.cholesky(c * P_pred)

    sigma = tf.concat([
        x_pred[None, :],
        x_pred[None, :] + tf.transpose(sqrtP),
        x_pred[None, :] - tf.transpose(sqrtP)
    ], axis=0)

    # --------------------------------------------------------
    # Nonlinear transformation
    # --------------------------------------------------------
    sigma_y = tf.vectorized_map(lambda s: h(s, t), sigma)

    y_pred = tf.reduce_sum(sigma_y * Wm[:, None], axis=0)

    dy = sigma_y - y_pred
    dx = sigma - x_pred

    # --------------------------------------------------------
    # Covariances
    # --------------------------------------------------------
    S = tf.einsum('i,ij,ik->jk', Wc, dy, dy) + R
    Pxz = tf.einsum('i,ij,ik->jk', Wc, dx, dy)

    tf.debugging.assert_all_finite(S, "Innovation covariance S contains NaNs/Infs")

    # --------------------------------------------------------
    # Kalman gain
    # --------------------------------------------------------
    K = tf.transpose(tf.linalg.solve(S, tf.transpose(Pxz)))

    v_t = y - y_pred

    # --------------------------------------------------------
    # State update
    # --------------------------------------------------------
    x_filt = x_pred + tf.linalg.matvec(K, v_t)

    # --------------------------------------------------------
    # Covariance update
    # --------------------------------------------------------
    P_filt = P_pred - K @ S @ tf.transpose(K)

    return x_filt, P_filt, v_t, S, K



@tf.function(reduce_retracing=True)
def _filter_core(
    Y,
    predict_fn,
    update_fn,
    R_mat,
    m0,
    P0,
    measurement_type,
    dtype
):
    # --------------------------------------------------------
    # Input validation 
    # --------------------------------------------------------
    if not tf.is_tensor(Y):
        raise TypeError("Y must be a TensorFlow tensor")

    if not tf.is_tensor(R_mat):
        raise TypeError("R_mat must be a TensorFlow tensor")

    if not tf.is_tensor(m0):
        raise TypeError("m0 must be a TensorFlow tensor")

    if not tf.is_tensor(P0):
        raise TypeError("P0 must be a TensorFlow tensor")

    if not callable(predict_fn):
        raise TypeError("predict_fn must be callable")

    if not isinstance(update_fn, dict):
        raise TypeError("update_fn must be a dictionary")

    if "h" not in update_fn or "step" not in update_fn:
        raise ValueError("update_fn must contain 'h' and 'step'")

    if len(Y.shape) != 2:
        raise ValueError("Y must be a 2D tensor (n_y, T)")

    if len(m0.shape) != 1:
        raise ValueError("m0 must be a 1D tensor")

    if len(P0.shape) != 2:
        raise ValueError("P0 must be a 2D tensor")

    if P0.shape[0] != P0.shape[1]:
        raise ValueError("P0 must be square")

    if m0.shape[0] != P0.shape[0]:
        raise ValueError("m0 and P0 dimension mismatch")
    
    Y = tf.cast(Y, dtype)
    R_mat = tf.cast(R_mat, dtype)
    m0 = tf.cast(m0, dtype)
    P0 = tf.cast(P0, dtype)

    # --------------------------------------------------------
    # Input checks
    # --------------------------------------------------------
    tf.debugging.assert_all_finite(Y, "Y contains NaN or Inf")
    tf.debugging.assert_all_finite(R_mat, "R_mat contains NaN or Inf")
    tf.debugging.assert_all_finite(m0, "m0 contains NaN or Inf")
    tf.debugging.assert_all_finite(P0, "P0 contains NaN or Inf")


    n_y, T = Y.shape
    n_x = m0.shape[0]
    
    # Allocation 
    mu_filt = tf.TensorArray(dtype, size=T)
    P_filt = tf.TensorArray(dtype, size=T)
    P_pred_store = tf.TensorArray(dtype, size=T)   

    # Initialisation
    x_filt = m0
    P_filt_t = P0
    loglik = tf.constant(0.0, dtype=dtype)

    # Recursion
    for t in tf.range(T):
        y_t = Y[:, t]

        # -------- PREDICT --------
        x_pred, P_pred = predict_fn(x_filt, P_filt_t, t)

        # STORE PREDICTED COVARIANCE
        P_pred_store = P_pred_store.write(t, P_pred)

        # -------- R handling --------
        # Gaussian case: use fixed observation covariance R_mat.
        # Poisson case, included to support the Li example C for particle flow particle filters:
        # observation variance equals the predicted intensity,
        # so the effective covariance is R_t = diag(h(x_pred, t)).
        if measurement_type == "poisson":
            y_pred = update_fn["h"](x_pred, t)
            R_foo = tf.linalg.diag(tf.reshape(y_pred, [-1]))
        else:
            R_foo = R_mat

        # -------- UPDATE --------
        x_filt, P_filt_t, v, S = update_fn["step"](x_pred, P_pred, y_t, R_foo, t)

        # -------- STORE --------
        mu_filt = mu_filt.write(t, x_filt)
        P_filt = P_filt.write(t, P_filt_t)

        # -------- LOG-LIK --------
        v_col = tf.reshape(v, (-1, 1))
        n_y_tf = tf.cast(tf.shape(Y)[0], dtype)
        log2pi = tf.math.log(tf.constant(2.0 * np.pi, dtype=dtype))

        ll = -0.5 * (n_y_tf * log2pi + tf.math.log(tf.linalg.det(S)) + tf.transpose(v_col) @ tf.linalg.solve(S, v_col))[0, 0]

        loglik += ll

    return {
        "mu_filt": tf.transpose(mu_filt.stack()),
        "P_filt": P_filt.stack(),
        "P_pred": P_pred_store.stack(),   
        "loglik": loglik
    }


def make_ukf_kernels(F_func, H_func, Q, alpha, beta, kappa):

    @tf.function(reduce_retracing=True)
    def predict_fn(x, P, t):
        dtype = x.dtype
        
        x = tf.cast(x, dtype)
        P = tf.cast(P, dtype)
        Q_ = tf.cast(Q, dtype)

        x_pred, P_pred = ukf_predict(x, P, F_func, Q_, alpha, beta, kappa, t)

        return tf.cast(x_pred, dtype), tf.cast(P_pred, dtype)

    @tf.function(reduce_retracing=True)
    def update_step(x_pred, P_pred, y, R, t):
        dtype = x_pred.dtype

        x_pred = tf.cast(x_pred, dtype)
        P_pred = tf.cast(P_pred, dtype)
        y = tf.cast(y, dtype)
        R_ = tf.cast(R, dtype)

        x, P, v, S, _ = ukf_update_check(
            x_pred, P_pred, y,
            H_func, R_,
            alpha, beta, kappa, t
        )

        return tf.cast(x, dtype), tf.cast(P, dtype), tf.cast(v, dtype), tf.cast(S, dtype)

    return predict_fn, {
        "step": update_step,
        "h": H_func
    }

# EKF

def make_ekf_kernels(F_func, H_func, F_jac, H_jac, Q):

    @tf.function(reduce_retracing=True)
    def predict_fn(x, P, t):
        dtype = x.dtype

        F = tf.cast(F_jac(x, t), dtype)
        x_pred = tf.cast(F_func(x, t), dtype)
        Q_ = tf.cast(Q, dtype)

        P_pred = F @ P @ tf.transpose(F) + Q_

        return x_pred, P_pred

    @tf.function(reduce_retracing=True)
    def update_step(x_pred, P_pred, y, R, t):
        dtype = x_pred.dtype

        H = tf.cast(H_jac(x_pred, t), dtype)
        y_pred = tf.cast(H_func(x_pred, t), dtype)
        R_ = tf.cast(R, dtype)
        
        # innovation
        v = y - y_pred
        S = H @ P_pred @ tf.transpose(H) + R_
        #kalman gain
        K = tf.transpose(tf.linalg.solve(S, tf.transpose(P_pred @ tf.transpose(H))))

        x = x_pred + tf.linalg.matvec(K, v)

        I = tf.eye(tf.shape(x_pred)[0], dtype=dtype)
        KH = K @ H
        # covariance
        P = (I - KH) @ P_pred @ tf.transpose(I - KH) + K @ R_ @ tf.transpose(K)

        return x, P, v, S

    return predict_fn, {
        "step": update_step,
        "h": H_func
    }

#### Experiments

In [ ]:
def run_kf_experiments_multivariate(
    y_tf,
    h_tf,
    phi,
    sigma_eps,
    sigma_eta,
    xi,
    dtype=tf.float64,
    m0=None,
    P0=None,
    alpha=1e-3,
    beta=2.0,
    kappa=0.0,
    eps_floor=1e-12,
    methods=None,
):
    """
    Multivariate EKF/UKF experiments for the SV model.

    Parameters
    ----------
    y_tf : tensor, shape (T,d)
    h_tf : tensor, shape (T,d)
    phi, sigma_eps, sigma_eta, xi : model parameters
    methods : list or None
        Subset of methods to run. Allowed values:
        - "EKF_misspec"
        - "UKF_misspec"
        - "EKF_correct"
        - "UKF_correct"

        If None, all four are run.

    Returns
    -------
    results_methods : dict
        Dictionary keyed by method name.
    """
    results_methods = {}

    if methods is None:
        methods = ["EKF_misspec", "UKF_misspec", "EKF_correct", "UKF_correct"]

    y_tf = tf.cast(y_tf, dtype)
    h_tf = tf.cast(h_tf, dtype)

    if len(y_tf.shape) != 2:
        raise ValueError("y_tf must be 2D, shape (T,d)")
    if len(h_tf.shape) != 2:
        raise ValueError("h_tf must be 2D, shape (T,d)")

    d = int(y_tf.shape[1])

    Y = tf.transpose(y_tf)   # (d,T)
    H_true = h_tf            # (T,d)

    Phi = _as_transition_matrix_tf(phi, d, dtype) 

    L_eta = _as_noise_chol_tf(sigma_eta, d, dtype, "sigma_eta")
    Q = L_eta @ tf.transpose(L_eta)

    sigma_eps_vec = _as_vector_param_tf(sigma_eps, d, dtype, "sigma_eps") 
    xi_vec = _as_vector_param_tf(xi, d, dtype, "xi")

    if m0 is None:
        m0 = tf.zeros([d], dtype=dtype)
    else:
        m0 = tf.cast(m0, dtype)

    if P0 is None:
        P0 = tf.eye(d, dtype=dtype)
    else:
        P0 = tf.cast(P0, dtype)

    # ============================================================
    # MISSPECIFIED CASE
    # ============================================================
    def F_func_mis(x, t):
        return tf.linalg.matvec(Phi, x)

    def F_jac_mis(x, t):
        return Phi

    def H_func_mis(x, t):
        return xi_vec * tf.exp(x / 2.0)

    def H_jac_mis(x, t):
        return tf.linalg.diag(0.5 * xi_vec * tf.exp(x / 2.0))

    R_mis = tf.linalg.diag(sigma_eps_vec**2)

    for method in ["EKF", "UKF"]:
        method_key = f"{method}_misspec"
        if method_key not in methods:
            continue

        print(f"  {method} (misspecified, multivariate)")

        if method == "EKF":
            predict_fn, update_fn = make_ekf_kernels(
                F_func=F_func_mis,
                H_func=H_func_mis,
                F_jac=F_jac_mis,
                H_jac=H_jac_mis,
                Q=Q
            )
        else:
            predict_fn, update_fn = make_ukf_kernels(
                F_func=F_func_mis,
                H_func=H_func_mis,
                Q=Q,
                alpha=alpha,
                beta=beta,
                kappa=kappa
            )

        out = _filter_core(
            Y=Y,
            predict_fn=predict_fn,
            update_fn=update_fn,
            R_mat=R_mis,
            m0=m0,
            P0=P0,
            measurement_type="gaussian",
            dtype=dtype
        )

        mu_filt = tf.transpose(out["mu_filt"])       # (T,d)
        mu_filt_mc = tf.expand_dims(mu_filt, axis=0) # (1,T,d)

        bias, rmse = compute_bias_rmse(H_true, mu_filt_mc)

        results_methods[method_key] = {
            "bias": bias,
            "rmse": rmse,
            "mu_filt": mu_filt,
            "P_filt": out["P_filt"],
            "P_pred": out["P_pred"],
            "loglik": out["loglik"]
        }

    # ============================================================
    # CORRECT CASE
    # ============================================================
    z_tf = tf.math.log(tf.maximum(y_tf**2, tf.constant(eps_floor, dtype=dtype)))
    Z = tf.transpose(z_tf)

    c_vec = tf.math.log((xi_vec**2) * (sigma_eps_vec**2)) - tf.constant(1.2704, dtype=dtype)

    def F_func_corr(x, t):
        return tf.linalg.matvec(Phi, x)

    def F_jac_corr(x, t):
        return Phi

    def H_func_corr(x, t):
        return x + c_vec

    def H_jac_corr(x, t):
        return tf.eye(d, dtype=dtype)

    R_corr = tf.eye(d, dtype=dtype) * tf.constant(np.pi**2 / 2.0, dtype=dtype)

    for method in ["EKF", "UKF"]:
        method_key = f"{method}_correct"
        if method_key not in methods:
            continue

        print(f"  {method} (correct, multivariate)")

        if method == "EKF":
            predict_fn, update_fn = make_ekf_kernels(
                F_func=F_func_corr,
                H_func=H_func_corr,
                F_jac=F_jac_corr,
                H_jac=H_jac_corr,
                Q=Q
            )
        else:
            predict_fn, update_fn = make_ukf_kernels(
                F_func=F_func_corr,
                H_func=H_func_corr,
                Q=Q,
                alpha=alpha,
                beta=beta,
                kappa=kappa
            )

        out = _filter_core(
            Y=Z,
            predict_fn=predict_fn,
            update_fn=update_fn,
            R_mat=R_corr,
            m0=m0,
            P0=P0,
            measurement_type="gaussian",
            dtype=dtype
        )

        mu_filt = tf.transpose(out["mu_filt"])       # (T,d)
        mu_filt_mc = tf.expand_dims(mu_filt, axis=0) # (1,T,d)

        bias, rmse = compute_bias_rmse(H_true, mu_filt_mc)

        results_methods[method_key] = {
            "bias": bias,
            "rmse": rmse,
            "mu_filt": mu_filt,
            "P_filt": out["P_filt"],
            "P_pred": out["P_pred"],
            "loglik": out["loglik"]
        }

    return results_methods


### BPF FOR EXECUTION
@tf.function(reduce_retracing=True)
def _run_bpf_mc_for_fixed_N(
    y_tf,
    h_tf,
    N,
    R,
    prop_fn,
    log_likelihood_fn,
    degeneracy_threshold,
    resampling_fn,
    resample_threshold,
    carry_resampled_weights,
    dtype
):
    """
    Run R Monte Carlo replications of the BPF for one fixed particle count N.

    Returns a dict of tensors.
    """
    y_tf = tf.cast(y_tf, dtype)
    h_tf = tf.cast(h_tf, tf.float64)

    if h_tf.shape.rank == 1:
        h_tf = tf.expand_dims(h_tf, axis=-1)

    T = tf.shape(y_tf)[0]
    d = tf.shape(y_tf)[1]

    est_ta = tf.TensorArray(tf.float64, size=R, infer_shape=False)
    ess_ta = tf.TensorArray(tf.float64, size=R, infer_shape=False)
    loglik_ta = tf.TensorArray(tf.float64, size=R)

    for r in tf.range(R):
        ests, ESSs, total_loglik = bpf_generic_resampling(
            Y=y_tf,
            Np=N,
            prop_fn=prop_fn,
            log_likelihood_fn=log_likelihood_fn,
            resampling_fn=resampling_fn,
            resample_threshold=resample_threshold,
            dtype=dtype,
            carry_resampled_weights=carry_resampled_weights,
        )

        est_ta = est_ta.write(r, tf.cast(ests, tf.float64))     # (T,d)
        ess_ta = ess_ta.write(r, tf.cast(ESSs, tf.float64))     # (T,)
        loglik_ta = loglik_ta.write(r, tf.cast(total_loglik, tf.float64))

    part_est_mc = est_ta.stack()      # (R,T,d)
    ESS_mc = ess_ta.stack()           # (R,T)
    loglik_mc = loglik_ta.stack()     # (R,)

    ESS_norm_mc = ESS_mc / tf.cast(N, tf.float64)

    ESS_mean_t = tf.reduce_mean(ESS_mc, axis=0)
    ESS_min_t = tf.reduce_min(ESS_mc, axis=0)

    ESS_norm_mean_t = tf.reduce_mean(ESS_norm_mc, axis=0)
    ESS_norm_min_t = tf.reduce_min(ESS_norm_mc, axis=0)

    bias, rmse = compute_bias_rmse(h_tf, part_est_mc)

    ess_mean = tf.reduce_mean(ESS_norm_mc)
    ess_min = tf.reduce_min(ESS_norm_mc)
    ess_frac_below = tf.reduce_mean(
        tf.cast(ESS_norm_mc < tf.cast(degeneracy_threshold, tf.float64), tf.float64)
    )
    ess_n_below = tf.reduce_sum(
        tf.cast(ESS_norm_mc < tf.cast(degeneracy_threshold, tf.float64), tf.int32)
    )
    loglik_mean = tf.reduce_mean(loglik_mc)

    return {
        "ESS_mean_t": ESS_mean_t,
        "ESS_min_t": ESS_min_t,
        "ESS_norm_mean_t": ESS_norm_mean_t,
        "ESS_norm_min_t": ESS_norm_min_t,
        "ESS_mc": ESS_mc,
        "ESS_norm_mc": ESS_norm_mc,
        "part_est_mc": part_est_mc,
        "loglikelihood_mc": loglik_mc,
        "bias": bias,
        "rmse": rmse,
        "ess_mean": ess_mean,
        "ess_min": ess_min,
        "ess_frac_below": ess_frac_below,
        "ess_n_below": ess_n_below,
        "loglik_mean": loglik_mean,
    }


def run_bpf_sv_experiments(
    phi,
    sigma_eta,
    sigma_eps,
    xi,
    Np,
    R,
    T=100,
    d=1,
    seed=123,
    degeneracy_threshold=0.2,
    resampling_fn=multinomial_resampling,
    resample_threshold=False,
    carry_resampled_weights=False,
    dtype=tf.float32
):
    if R <= 0:
        raise ValueError("R must be > 0")
    if T <= 0:
        raise ValueError("T must be > 0")
    if d <= 0:
        raise ValueError("d must be > 0")

    # --------------------------------------------------
    # STEP 1: simulate data
    # --------------------------------------------------
    sim_out = SV_model_sim_tf_h(
        iT=T,
        phi=phi,
        sigma_eta=sigma_eta,
        sigma_eps=sigma_eps,
        xi=xi,
        seed=seed,
        d=d,
        dtype=tf.float64
    )

    y_tf = tf.cast(sim_out["vY"], dtype)
    h_tf = tf.cast(sim_out["h"], tf.float64)

    if y_tf.shape.rank == 1:
        y_tf = tf.expand_dims(y_tf, axis=-1)
    if h_tf.shape.rank == 1:
        h_tf = tf.expand_dims(h_tf, axis=-1)

    # --------------------------------------------------
    # STEP 2: build filter components
    # --------------------------------------------------
    prop_fn = make_prop_sv(phi=phi, sigma_eta=sigma_eta, dtype=dtype)
    loglik_fn = make_loglik_sv(sigma_eps=sigma_eps, xi=xi, dtype=dtype)

    # --------------------------------------------------
    # STEP 3: run experiments for each N
    # --------------------------------------------------
    results_BPF = {}
    metrics = {}

    for N in Np:
        print(f"  N = {N}")

        outN = _run_bpf_mc_for_fixed_N(
            y_tf=y_tf,
            h_tf=h_tf,
            N=int(N),
            R=int(R),
            prop_fn=prop_fn,
            log_likelihood_fn=loglik_fn,
            degeneracy_threshold=tf.cast(degeneracy_threshold, tf.float64),
            resampling_fn=resampling_fn,
            resample_threshold=resample_threshold,
            carry_resampled_weights=carry_resampled_weights,
            dtype=dtype
        )

        results_BPF[N] = {
            "ESS_mean_t": outN["ESS_mean_t"],
            "ESS_min_t": outN["ESS_min_t"],
            "ESS_norm_mean_t": outN["ESS_norm_mean_t"],
            "ESS_norm_min_t": outN["ESS_norm_min_t"],
            "ESS_mc": outN["ESS_mc"],
            "ESS_norm_mc": outN["ESS_norm_mc"],
            "part_est_mc": outN["part_est_mc"],
            "loglikelihood_mc": outN["loglikelihood_mc"],
        }

        metrics[N] = {
            "bias": outN["bias"],
            "rmse": outN["rmse"],
            "ess_mean": outN["ess_mean"],
            "ess_min": outN["ess_min"],
            "ess_frac_below": outN["ess_frac_below"],
            "ess_n_below": outN["ess_n_below"],
            "loglik_mean": outN["loglik_mean"],
        }

    return {
        "sim": {
            "y_tf": y_tf,
            "h_tf": h_tf
        },
        "ESS": results_BPF,
        "metrics": metrics
    }


def compare_methods_one_config(
    d,
    phi,
    sigma_eps,
    sigma_eta=1.0,
    xi=1.0,
    Np=(5, 10, 20),
    T=10,
    N_MC=3,
    seed=123,
    dtype_bpf=tf.float32,
    dtype_kf=tf.float64,
    sample_interval=0.01,
):
    # ---------- simulate once ----------
    sim_out = SV_model_sim_tf_h(
        iT=T,
        phi=phi,
        sigma_eta=sigma_eta,
        sigma_eps=sigma_eps,
        xi=xi,
        seed=seed,
        d=d,
        dtype=tf.float64
    )

    y_tf = tf.cast(sim_out["vY"], tf.float64)
    h_tf = tf.cast(sim_out["h"], tf.float64)

    if y_tf.shape.rank == 1:
        y_tf = tf.expand_dims(y_tf, axis=-1)
    if h_tf.shape.rank == 1:
        h_tf = tf.expand_dims(h_tf, axis=-1)

    # ---------- BPF ----------
    prop_fn = make_prop_sv(phi=phi, sigma_eta=sigma_eta, dtype=dtype_bpf)
    loglik_fn = make_loglik_sv(sigma_eps=sigma_eps, xi=xi, dtype=dtype_bpf)

    ess_out = {}
    metrics_out = {}
    benchmark_out = {"BPF": {}, "KF": {}}

    for N in Np:
        outN, statsN = benchmark_cpu(
            _run_bpf_mc_for_fixed_N,
            y_tf=tf.cast(y_tf, dtype_bpf),
            h_tf=tf.cast(h_tf, tf.float64),
            N=int(N),
            R=int(N_MC),
            prop_fn=prop_fn,
            log_likelihood_fn=loglik_fn,
            degeneracy_threshold=tf.constant(0.2, tf.float64),
            resampling_fn=multinomial_resampling,
            resample_threshold=False,
            carry_resampled_weights=False,
            dtype=dtype_bpf,
            sample_interval=sample_interval,
        )

        ess_out[N] = {
            "ESS_mean_t": outN["ESS_mean_t"],
            "ESS_min_t": outN["ESS_min_t"],
            "ESS_norm_mean_t": outN["ESS_norm_mean_t"],
            "ESS_norm_min_t": outN["ESS_norm_min_t"],
#            "ESS_mc": outN["ESS_mc"],
#            "ESS_norm_mc": outN["ESS_norm_mc"],
#            "part_est_mc": outN["part_est_mc"],
#            "loglikelihood_mc": outN["loglikelihood_mc"],
        }

        metrics_out[N] = {
            "bias": outN["bias"],
            "rmse": outN["rmse"],
            "ess_mean": outN["ess_mean"],
            "ess_min": outN["ess_min"],
            "ess_frac_below": outN["ess_frac_below"],
            "ess_n_below": outN["ess_n_below"],
            "loglik_mean": outN["loglik_mean"],
        }

        benchmark_out["BPF"][N] = statsN

        print(
            f"BPF N={N} | runtime={statsN['runtime_seconds']:.4f}s | "
            f"peak+={statsN['rss_peak_increase_mb']:.2f} MB"
        )

    # ---------- EKF / UKF ----------
    methods_list = ["EKF_misspec", "UKF_misspec", "EKF_correct", "UKF_correct"]
    kf_out = {}

    for method_name in methods_list:
        out_method, stats_method = benchmark_cpu(
            run_kf_experiments_multivariate,
            y_tf=tf.cast(y_tf, dtype_kf),
            h_tf=tf.cast(h_tf, dtype_kf),
            phi=phi,
            sigma_eps=sigma_eps,
            sigma_eta=sigma_eta,
            xi=xi,
            dtype=dtype_kf,
            methods=[method_name],
            sample_interval=sample_interval,
        )

        kf_out.update(out_method)
        benchmark_out["KF"][method_name] = stats_method

        print(
            f"{method_name} | runtime={stats_method['runtime_seconds']:.4f}s | "
            f"peak+={stats_method['rss_peak_increase_mb']:.2f} MB"
        )

    return {
        "sim": {"y_tf": y_tf, "h_tf": h_tf},
        "ESS": ess_out,
        "metrics": metrics_out,
        "KF": kf_out,
        "benchmark": benchmark_out,
    }

In [ ]:
# Helper
def compute_bias_rmse(h_true, est_mc):
    """
    h_true:  (T,) or (T,d)
    est_mc:  (R,T) or (R,T,d)
    """
    h_true = tf.cast(tf.convert_to_tensor(h_true), tf.float64)
    est_mc = tf.cast(tf.convert_to_tensor(est_mc), tf.float64)

    if h_true.shape.rank == 1:
        h_true = tf.expand_dims(h_true, axis=-1)
    if est_mc.shape.rank == 2:
        est_mc = tf.expand_dims(est_mc, axis=-1)

    bias = tf.reduce_mean(est_mc, axis=0) - h_true
    rmse = tf.sqrt(tf.reduce_mean((est_mc - h_true) ** 2, axis=0))

    return bias, rmse



def benchmark_cpu(func, *args, sample_interval=0.01, **kwargs):
    import time
    import threading
    import psutil

    process = psutil.Process()

    rss_before = process.memory_info().rss
    peak_rss = {"value": rss_before}
    stop_event = threading.Event()

    def monitor():
        peak = rss_before
        while not stop_event.is_set():
            rss = process.memory_info().rss
            if rss > peak:
                peak = rss
            time.sleep(sample_interval)
        peak_rss["value"] = peak

    thread = threading.Thread(target=monitor, daemon=True)
    thread.start()

    t0 = time.perf_counter()
    out = func(*args, **kwargs)
    t1 = time.perf_counter()

    stop_event.set()
    thread.join()

    rss_after = process.memory_info().rss

    stats = {
        "runtime_seconds": t1 - t0,
        "rss_before_mb": rss_before / (1024**2),
        "rss_after_mb": rss_after / (1024**2),
        "rss_peak_mb": peak_rss["value"] / (1024**2),
        "rss_delta_mb": (rss_after - rss_before) / (1024**2),
        "rss_peak_increase_mb": (peak_rss["value"] - rss_before) / (1024**2),
    }

    return out, stats


def plot_benchmark_grouped(results_all):
    import matplotlib.pyplot as plt
    import numpy as np

    d_values = sorted({cfg[0] for cfg in results_all.keys()})

    first_cfg = next(iter(results_all))
    method_names = []

    for N in sorted(results_all[first_cfg]["benchmark"]["BPF"].keys()):
        method_names.append(f"BPF N={N}")

    for name in sorted(results_all[first_cfg]["benchmark"]["KF"].keys()):
        method_names.append(name)

    cpu_by_method = {m: [] for m in method_names}
    time_by_method = {m: [] for m in method_names}

    for d in d_values:
        cfgs_d = [cfg for cfg in results_all if cfg[0] == d]

        for m in method_names:
            cpu_vals = []
            time_vals = []

            for cfg in cfgs_d:
                bench = results_all[cfg]["benchmark"]

                if m.startswith("BPF N="):
                    N = int(m.replace("BPF N=", ""))
                    s = bench["BPF"][N]
                else:
                    s = bench["KF"][m]

                cpu_vals.append(s["rss_peak_mb"])
                time_vals.append(s["runtime_seconds"])

            cpu_by_method[m].append(np.mean(cpu_vals))
            time_by_method[m].append(np.mean(time_vals))

    x = np.arange(len(d_values))
    n_methods = len(method_names)
    width = 0.8 / n_methods

    # Plot style: 2 rows, 1 column
    fig, axes = plt.subplots(2, 1, figsize=(10, 10)) 

    for j, m in enumerate(method_names):
        offset = (j - (n_methods - 1) / 2) * width

        axes[0].bar(x + offset, cpu_by_method[m], width=width, label=m)
        axes[1].bar(x + offset, time_by_method[m], width=width, label=m)

    # --- CPU ---
    axes[0].set_title("CPU peak memory")
    axes[0].set_ylabel("MB")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(d_values)
    axes[0].grid(axis="y", alpha=0.3)

    # --- Runtime ---
    axes[1].set_title("Runtime")
    axes[1].set_ylabel("seconds")
    axes[1].set_xlabel("State dimension d")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(d_values)
    axes[1].grid(axis="y", alpha=0.3)

    # legend on top
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center",
               ncol=min(len(labels), 4), frameon=False)

    fig.subplots_adjust(top=0.88, hspace=0.3)

    plt.show()


def plot_ESS_heatmap_over_d(
    results_all,
    N_list,
    phi,
    sigma_eps,
    use_min=False,
):
    """
    Heatmap of normalized ESS over time for fixed (phi, sigma_eps),
    with one row per state dimension d and one column per N.

    Parameters
    ----------
    results_all : dict
        Dictionary keyed by (d, phi, sigma_eps).
    N_list : list
        Particle counts to plot.
    phi : float
        Fixed phi value.
    sigma_eps : float
        Fixed sigma_eps value.
    use_min : bool
        If True, use ESS_norm_min_t.
        If False, use ESS_norm_mean_t.
    """
    import matplotlib.pyplot as plt
    import tensorflow as tf
    import numpy as np

    configs = sorted(
        [cfg for cfg in results_all if cfg[1] == phi and cfg[2] == sigma_eps],
        key=lambda x: x[0]
    )

    if not configs:
        raise ValueError(f"No configurations found for phi={phi}, sigma_eps={sigma_eps}.")

    d_labels = [str(d) for (d, _, _) in configs]
    ess_key = "ESS_min_t" if use_min else "ESS_mean_t"

    fig, axes = plt.subplots(
        1, len(N_list),
        figsize=(5 * len(N_list), max(4, 0.5 * len(configs) + 2)),
        sharey=True
    )

    if len(N_list) == 1:
        axes = [axes]

    im = None

    for ax, N in zip(axes, N_list):
        H = tf.stack(
            [tf.cast(results_all[cfg]["ESS"][N][ess_key], tf.float64) for cfg in configs],
            axis=0
        ).numpy()

        im = ax.imshow(H, aspect="auto", origin="lower", vmin=0.0, vmax=1.0)
        ax.set_title(f"N = {N}")
        ax.set_xlabel("Time")

    axes[0].set_yticks(np.arange(len(configs)))
    axes[0].set_yticklabels(d_labels)
    axes[0].set_ylabel("State dimension d")

    for ax in axes[1:]:
        ax.set_yticks(np.arange(len(configs)))
        ax.set_yticklabels([])

    fig.subplots_adjust(right=0.88, wspace=0.15)
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    fig.colorbar(im, cax=cax, label="ESS / N")

    title_metric = "minimum" if use_min else "mean"
    fig.suptitle(
        f"Normalized {title_metric} ESS heatmap for fixed φ={phi}, σe={sigma_eps}",
        y=0.98
    )

    plt.show()


def plot_ESS_over_time_bpf(
    results_all,
    phi_fixed,
    sigma_eps_fixed,
    d_list=None,
    N_bpf_list=None,
    use_min=False,
    figsize_per_row=5,
):
    import matplotlib.pyplot as plt
    import tensorflow as tf
    import numpy as np

    configs = sorted(
        [
            (d, phi, sigma_eps)
            for (d, phi, sigma_eps) in results_all.keys()
            if phi == phi_fixed and sigma_eps == sigma_eps_fixed
        ],
        key=lambda x: x[0]
    )

    if not configs:
        raise ValueError(f"No configurations found for phi={phi_fixed}, sigma_eps={sigma_eps_fixed}")

    if d_list is None:
        d_list = [d for (d, _, _) in configs]

    d_list = sorted(d_list)

    if N_bpf_list is None:
        first_cfg = (d_list[0], phi_fixed, sigma_eps_fixed)
        N_bpf_list = sorted(results_all[first_cfg]["ESS"].keys())

    ess_key = "ESS_min_t" if use_min else "ESS_mean_t"

    palette = ["#00cc96", "#ffa600", "#ab63fa", "#19d3f3"]
    bpf_colors = {
        N: palette[i % len(palette)] for i, N in enumerate(sorted(N_bpf_list))
    }

    fig, axes = plt.subplots(
        len(d_list), 1,
        figsize=(8, figsize_per_row * len(d_list)),
        squeeze=False
    )

    for i, d in enumerate(d_list):
        ax = axes[i, 0]
        res = results_all[(d, phi_fixed, sigma_eps_fixed)]

        for N in N_bpf_list:
            y = tf.cast(res["ESS"][N][ess_key], tf.float64).numpy()
            x = np.arange(len(y))

            ax.plot(
                x, y,
                linewidth=2.5,
                color=bpf_colors[N],
                label=f"BPF N={N}"
            )

        ax.set_title(f"d = {d}", fontsize=12)
        ax.set_xlabel("Time")
        ax.set_ylabel("ESS")
        ax.grid(alpha=0.3)
        ax.legend(frameon=False, fontsize=9, ncol=2)

    title_metric = "minimum" if use_min else "mean"
    fig.suptitle(
        f"{title_metric} ESS over time | φ={phi_fixed}, σe={sigma_eps_fixed}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


def plot_metric_over_time_algorithms(
    results_all,
    phi_fixed,
    sigma_eps_fixed,
    d_list=None,
    metric="RMSE",
    N_bpf_list=None,
    reduce_mode="mean",
    figsize_per_row=5,
):
    import matplotlib.pyplot as plt
    import tensorflow as tf
    import numpy as np

    metric = metric.upper()

    configs = sorted(
        [
            (d, phi, sigma_eps)
            for (d, phi, sigma_eps) in results_all.keys()
            if phi == phi_fixed and sigma_eps == sigma_eps_fixed
        ],
        key=lambda x: x[0]
    )

    if d_list is None:
        d_list = [d for (d, _, _) in configs]

    d_list = sorted(d_list)

    # Colors
    kf_colors = {
        "EKF_misspec": "#ff0054",  
        "EKF_correct": "#ff0054",
        "UKF_misspec": "#0096ff",   
        "UKF_correct": "#0096ff",
    }

    # BPF 
    if N_bpf_list is not None:
        palette = ["#00cc96", "#ffa600", "#ab63fa", "#19d3f3"]  
        bpf_colors = {
            N: palette[i % len(palette)] for i, N in enumerate(sorted(N_bpf_list))
        }
    else:
        bpf_colors = {}

    fig, axes = plt.subplots(
        len(d_list), 1,
        figsize=(8, figsize_per_row * len(d_list)),
        squeeze=False
    )

    def reduce_series(arr):
        arr = np.asarray(arr)
        if arr.ndim == 1:
            return arr
        if reduce_mode == "mean":
            return np.mean(arr, axis=-1)
        if reduce_mode == "median":
            return np.median(arr, axis=-1)
        if reduce_mode == "max":
            return np.max(arr, axis=-1)
        raise ValueError("invalid reduce_mode")

    for i, d in enumerate(d_list):
        ax = axes[i, 0]
        res = results_all[(d, phi_fixed, sigma_eps_fixed)]

        # ---------- BPF ----------
        if N_bpf_list is not None:
            for N in N_bpf_list:
                if metric == "RMSE":
                    arr = tf.cast(res["metrics"][N]["rmse"], tf.float64).numpy()
                else:
                    arr = tf.cast(res["metrics"][N]["nees"], tf.float64).numpy()

                y = reduce_series(arr)
                x = np.arange(len(y))

                ax.plot(
                    x, y,
                    linewidth=2.5,
                    color=bpf_colors[N],
                    label=f"BPF N={N}"
                )

        # ---------- KF ----------
        for method_name, out in res["KF"].items():

            linestyle = "-" if "correct" in method_name else "--"
            color = kf_colors.get(method_name, "black")

            if metric == "RMSE":
                arr = tf.cast(out["rmse"], tf.float64).numpy()
            else:
                arr = tf.cast(out["nees"], tf.float64).numpy()

            y = reduce_series(arr)
            x = np.arange(len(y))

            ax.plot(
                x, y,
                linewidth=2.5,
                linestyle=linestyle,
                color=color,
                label=method_name.replace("_", " ")
            )

        ax.set_title(f"d = {d}", fontsize=12)
        ax.set_xlabel("Time")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)

        # legend
        ax.legend(frameon=False, fontsize=9, ncol=2)

    fig.suptitle(
        f"{metric} over time | φ={phi_fixed}, σe={sigma_eps_fixed}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()


def plot_rmse_over_dimension(
    results_all,
    phi_fixed,
    sigma_eps_fixed,
    N_list,
    reduce_time="mean",
    reduce_coord="mean",
):
    """
    Plot RMSE as a function of state dimension d.

    Parameters
    ----------
    results_all : dict
        Dictionary keyed by (d, phi, sigma_eps).
    phi_fixed : float
        Fixed phi value.
    sigma_eps_fixed : float
        Fixed sigma_eps value.
    N_list : list
        BPF particle counts to include.
    reduce_time : str
        How to reduce over time:
        - "mean"
        - "median"
        - "max"
        - "final"
    reduce_coord : str
        How to reduce over coordinates if RMSE has shape (T,d):
        - "mean"
        - "median"
        - "max"
    """
    import matplotlib.pyplot as plt
    import tensorflow as tf
    import numpy as np

    configs = sorted(
        [
            (d, phi, sigma_eps)
            for (d, phi, sigma_eps) in results_all.keys()
            if phi == phi_fixed and sigma_eps == sigma_eps_fixed
        ],
        key=lambda x: x[0]
    )

    if not configs:
        raise ValueError(
            f"No configurations found for phi={phi_fixed}, sigma_eps={sigma_eps_fixed}."
        )

    d_values = [d for (d, _, _) in configs]

    def reduce_rmse(arr):
        arr = np.asarray(arr)

        if arr.ndim == 2:
            if reduce_coord == "mean":
                arr = np.mean(arr, axis=-1)
            elif reduce_coord == "median":
                arr = np.median(arr, axis=-1)
            elif reduce_coord == "max":
                arr = np.max(arr, axis=-1)
            else:
                raise ValueError("reduce_coord must be 'mean', 'median', or 'max'.")

        if reduce_time == "mean":
            return np.mean(arr)
        elif reduce_time == "median":
            return np.median(arr)
        elif reduce_time == "max":
            return np.max(arr)
        elif reduce_time == "final":
            return arr[-1]
        else:
            raise ValueError("reduce_time must be 'mean', 'median', 'max', or 'final'.")

    plt.figure(figsize=(8, 5))

    # ---------------- BPF lines ----------------
    for N in N_list:
        y_vals = []

        for cfg in configs:
            rmse = tf.cast(results_all[cfg]["metrics"][N]["rmse"], tf.float64).numpy()
            y_vals.append(reduce_rmse(rmse))

        plt.plot(d_values, y_vals, marker="o", linewidth=2, label=f"BPF N={N}")

    # ---------------- KF lines ----------------
    kf_methods = ["EKF_misspec", "UKF_misspec", "EKF_correct", "UKF_correct"]

    for method in kf_methods:
        y_vals = []

        for cfg in configs:
            rmse = tf.cast(results_all[cfg]["KF"][method]["rmse"], tf.float64).numpy()
            y_vals.append(reduce_rmse(rmse))

        plt.plot(d_values, y_vals, marker="o", linewidth=2, linestyle="--", label=method)

    plt.xlabel("State dimension d")
    plt.ylabel("RMSE")
    plt.title(
        f"RMSE over dimension for fixed φ={phi_fixed}, σe={sigma_eps_fixed}"
    )
    plt.grid(alpha=0.25)
    plt.legend(frameon=False, ncol=2)
    plt.show()


def plot_ESS_heatmap_over_d(
    results_all,
    N_list,
    phi,
    sigma_eps,
    use_min=False,
):
    import matplotlib.pyplot as plt
    import tensorflow as tf
    import numpy as np

    # -------- FILTER CONFIGS --------
    configs = sorted(
        [cfg for cfg in results_all if cfg[1] == phi and cfg[2] == sigma_eps],
        key=lambda x: x[0]
    )

    if not configs:
        raise ValueError(f"No configurations found for phi={phi}, sigma_eps={sigma_eps}.")

    d_labels = [str(d) for (d, _, _) in configs]
    ess_key = "ESS_min_t" if use_min else "ESS_mean_t"

    # -------- BUILD HEATMAP DATA --------
    H_list = []
    for N in N_list:
        H = tf.stack(
            [tf.cast(results_all[cfg]["ESS"][N][ess_key], tf.float64) for cfg in configs],
            axis=0
        ).numpy()
        H_list.append(H)

    # -------- GLOBAL COLOR SCALE --------
    all_values = np.concatenate([H.ravel() for H in H_list])
    vmin = np.nanmin(all_values)
    vmax = np.nanmax(all_values)

    # -------- FIGURE (FIXED HEIGHT) --------
    fig, axes = plt.subplots(
        1, len(N_list),
        figsize=(6 * len(N_list), 4.5),
        sharey=True
    )

    if len(N_list) == 1:
        axes = [axes]

    im = None

    # -------- PLOT --------
    for ax, N, H in zip(axes, N_list, H_list):
        im = ax.imshow(
            H,
            aspect="auto",
            origin="lower",
            vmin=vmin,
            vmax=vmax,
            cmap="viridis"
        )
        ax.set_title(f"N = {N}")
        ax.set_xlabel("Time")

    # -------- Y AXIS ONLY LEFT --------
    axes[0].set_yticks(np.arange(len(configs)))
    axes[0].set_yticklabels(d_labels)
    axes[0].set_ylabel("State dimension d")

    for ax in axes[1:]:
        ax.tick_params(axis="y", labelleft=False, left=False)

    # -------- COLORBAR --------
    fig.subplots_adjust(left=0.15, right=0.88, wspace=0.25)
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    fig.colorbar(im, cax=cax, label="ESS")

    # -------- TITLE --------
    title_metric = "minimum" if use_min else "mean"
    fig.suptitle(
        f"{title_metric} ESS heatmap | φ={phi}, σe={sigma_eps}",
        y=0.98
    )

    plt.show()


def build_ESS_summary_dataframe(results_all):
    rows = []

    for (d, phi, sigma_eps), res in results_all.items():
        for N, met in res["metrics"].items():
            row = {
                "d": int(d),
                "phi": phi,
                "sigma_eps": sigma_eps,
                "N": int(N),
                "n_below_thr": int(met["ess_n_below"].numpy()),
                "frac_below_thr": float(met["ess_frac_below"].numpy()),
                "mean_ESS_norm": float(met["ess_mean"].numpy()),
#                "q05_ESS_norm": float(met["ess_q05"].numpy()),
                "min_ESS_norm": float(met["ess_min"].numpy()),
                "loglik_mean": float(met["loglik_mean"].numpy()),
                "rmse": float(tf.reduce_mean(tf.cast(met["rmse"], tf.float64)).numpy())
            }
            rows.append(row)

    return pd.DataFrame(rows)

COMPARING EKF/UKF AND BPF 

In [ ]:
# RUN EXPERIMENTS - QUICK RUN
results_all = {}

for d in [10]: 
    for phi in [0.5]: 
        for sigma_eps in [1.0]:
            print(f"\nRunning d={d}, phi={phi}, sigma_eps={sigma_eps}")

            results_all[(d, phi, sigma_eps)] = compare_methods_one_config(
                d=d,
                phi=phi,
                sigma_eps=sigma_eps,
                sigma_eta=1.0,
                xi=1.0,
                Np=(50, 60, 80),
                T=40,
                N_MC=20,
                dtype_bpf=tf.float32,
                dtype_kf=tf.float64,
            )

In [ ]:
# RUN EXPERIMENTS 
results_all = {}

for d in [5, 20, 64]:
    for phi in [0.5, 0.95]:
        for sigma_eps in [0.3, 2.0]:
            print(f"\nRunning d={d}, phi={phi}, sigma_eps={sigma_eps}")

            results_all[(d, phi, sigma_eps)] = compare_methods_one_config(
                d=d,
                phi=phi,
                sigma_eps=sigma_eps,
                sigma_eta=1.0,
                xi=1.0,
                Np=(50, 150, 400),
                T=100,
                N_MC=60,
                dtype_bpf=tf.float32,
                dtype_kf=tf.float64,
            )

In [ ]:
# Log-likelihood comparison

for cfg, res in results_all.items():
    d, phi, sigma_eps = cfg

    print(f"\n=== d={d}, phi={phi}, sigma_eps={sigma_eps} ===")

    # BPF
    for N in res["metrics"]:
        print(f"BPF N={N}: loglik_mean = {res['metrics'][N]['loglik_mean']:.4f}")

    # EKF / UKF
    for method in res["KF"]:
        ll = res["KF"][method]["loglik"].numpy()
        print(f"{method}: loglik = {ll:.4f}")
        

In [ ]:
####### PLOTS
plot_ESS_heatmap_over_d(
    results_all,
    N_list=[50, 150, 400],
    phi=0.95,
    sigma_eps=0.3,
    use_min=False
)

plot_ESS_over_time_bpf(
    results_all,
    phi_fixed=0.95,
    sigma_eps_fixed=0.3,
    N_bpf_list=[50, 150, 400],
    use_min=False,
)



plot_metric_over_time_algorithms(
    results_all,
    phi_fixed=0.95,
    sigma_eps_fixed=0.3,
    d_list=[5, 15, 60],
    metric="RMSE", 
    N_bpf_list=[50, 150, 400],
    reduce_mode="mean",
)

plot_rmse_over_dimension(
    results_all,
    phi_fixed=0.95,
    sigma_eps_fixed=0.3,
    N_list=[50, 150, 400],
    reduce_time="mean",
    reduce_coord="mean",
)

plot_benchmark_grouped(results_all)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

phi_list = [0.5, 0.95]
N_list = [50, 150, 400]
d_list = [5, 15, 64]
sigma_eps = 0.3

for phi in phi_list:
    phi_tag = str(phi).replace(".", "p")

    plot_ESS_heatmap_over_d(
        results_all,
        N_list=N_list,
        phi=phi,
        sigma_eps=sigma_eps,
        use_min=False,
    )
    plt.savefig(f"ESS_heatmap_phi{phi_tag}.png", dpi=300, bbox_inches="tight")
    plt.close("all")

    # ------------------------
    # OVER-TIME PLOTS (SPLIT BY d)
    # ------------------------
    for d in d_list:
        d_tag = f"d{d}"

        # ESS over time
        plot_ESS_over_time_bpf(
            results_all,
            phi_fixed=phi,
            sigma_eps_fixed=sigma_eps,
            d_list=[d],  
            N_bpf_list=N_list,
            use_min=False,
        )
        plt.savefig(f"ESS_time_{d_tag}_phi{phi_tag}.png", dpi=300, bbox_inches="tight")
        plt.close("all")

        # RMSE over time
        plot_metric_over_time_algorithms(
            results_all,
            phi_fixed=phi,
            sigma_eps_fixed=sigma_eps,
            d_list=[d],  
            metric="RMSE",
            N_bpf_list=N_list,
            reduce_mode="mean",
        )
        plt.savefig(f"RMSE_time_{d_tag}_phi{phi_tag}.png", dpi=300, bbox_inches="tight")
        plt.close("all")

    plot_rmse_over_dimension(
        results_all,
        phi_fixed=phi,
        sigma_eps_fixed=sigma_eps,
        N_list=N_list,
        reduce_time="mean",
        reduce_coord="mean",
    )
    plt.savefig(f"RMSE_dim_phi{phi_tag}.png", dpi=300, bbox_inches="tight")
    plt.close("all")

#### EKF/UKF - Diagnostic Analysis 
Linearization accuracy limits and sigma-points failures

In [ ]:
def compute_jacobian_tf(fun, x, tol=1e-6):
    """
    Compute numerical derivative of fun at x using central difference.

    Input:
    ------
    fun : callable, function to differentiate
    x : float or tf.Tensor, point where derivative is evaluated 
    tol : float, small perturbation for finite difference (optional, set to 1e-6)

    Output:
    ------
        J : float or tf.Tensor, numerical derivative of fun at x
    """
    # Check float64
    if isinstance(x, tf.Tensor):
        x = tf.cast(x, dtype=tf.float64)
    else:
        x = tf.convert_to_tensor(x, dtype=tf.float64)

    # Check finiteness
    if not tf.math.is_finite(x):
        raise ValueError("Jacobian: x not finite")
    f1 = tf.convert_to_tensor(fun(x + tol), dtype=tf.float64)
    f2 = tf.convert_to_tensor(fun(x - tol), dtype=tf.float64)
    
    # Check function outputs finite
    if not (tf.math.is_finite(f1) and tf.math.is_finite(f2)):
        raise ValueError("Jacobian: function returned non-finite values")

    # Compute Numerical derivative
    J = (f1 - f2) / (2.0 * tol)

    # Check derivative is finite
    if not tf.math.is_finite(J):
        raise ValueError("Jacobian: non-finite derivative")
        
    # Check derivative is not numerically zero
    if tf.abs(J) < 1e-8:
        raise ValueError(f"Jacobian numerically zero at x = {x.numpy()}")

    return J


def compute_sigma_points_tf(mu, P, alpha=0.5, beta=2.0, kappa=0.0):
    """
    Computes sigma points and corresponding weights for the Unscented Transform
    Operates on a single scalar state; for multivariate states, apply independently per dimension.

    Input:
    ------
    mu : float or tf.Tensor, mean of the state
    P : float or tf.Tensor, variance of the state 
    alpha : float, spread scaling parameter (optional, set to 0.5)
    beta : float, distribution prior info parameter (optional, set to 2.0)
    kappa : float, secondary scaling parameter, (optional, set to 0.0)

    Output:
    ------
    dict: 
        "X" : tf.Tensor, sigma points [mu, mu+spread, mu-spread]  (shape [3])
        "Wm" :  tf.Tensor, weights for mean  (shape [3])
        "Wc" tf.Tensor, weights for covariance (shape [3])
    """
    
    # Conversion 
    mu = tf.convert_to_tensor(mu, dtype=tf.float64)
    P = tf.convert_to_tensor(P, dtype=tf.float64)
    alpha = tf.convert_to_tensor(alpha, dtype=tf.float64)
    beta = tf.convert_to_tensor(beta, dtype=tf.float64)
    kappa = tf.convert_to_tensor(kappa, dtype=tf.float64)

    # Check input 
    if not tf.math.is_finite(mu):
        raise ValueError("Sigma points: mu not finite")
    if not tf.math.is_finite(P) or P <= 0:
        raise ValueError("Sigma points: P must be positive")

    n = 1
    lambda_ = alpha**2 * (n + kappa) - n
    denom = n + lambda_

    if denom <= 0 or not tf.math.is_finite(denom):
        raise ValueError("Invalid scaling: n + lambda <= 0")

    gamma = tf.sqrt(denom)
    spread = gamma * tf.sqrt(P)

    # Robustness checks
    if not tf.math.is_finite(spread) or spread <= 0:
        raise ValueError("Sigma points collapsed: spread <= 0")
    # 
    if spread < 1e-14 * (tf.abs(mu) + 1):
        raise ValueError("Sigma points numerically degenerate (spread too small)")

    # Compute sigma points: in the univariate case three points are needed 
    X = tf.stack([mu, mu + spread, mu - spread])

    # Check degeneracy 
    tol = 1e-12
    if tf.reduce_max(tf.abs(X - X[0])) < tol:
        raise ValueError("Sigma points degenerate: all points coincide")

    # Compute weights
    Wm = tf.stack([lambda_ / denom, 1.0/(2*denom), 1.0/(2*denom)])
    Wc = tf.identity(Wm)
    Wc = tf.tensor_scatter_nd_add(Wc, [[0]], [1 - alpha**2 + beta])

    # Check weights finite
    if not tf.reduce_all(tf.math.is_finite(Wm)) or not tf.reduce_all(tf.math.is_finite(Wc)):
        raise ValueError("Non-finite UKF weights")

    return {"X": X, "Wm": Wm, "Wc": Wc}



def EKF_UKF_univariate(
    y,
    f_fun,
    h_fun,
    sigma_eta,
    sigma_e,
    m0=0.0,
    P0=1.0,
    method="EKF",
    return_diagnostics=False,
    stop_on_failure=True,
    x_tol=0.05,              # chosen "bad geometry" threshold
    crosscov_tol=1e-10,      # UKF sigma-point degeneracy threshold
    jf_fun=None,
    jh_fun=None
):
    """
    Univariate EKF / UKF with diagnostics.

    Preferred bad-geometry diagnostic:
        abs(mu_pred_t) < x_tol

    For UKF, we also track sigma-point degeneracy:
        abs(C_xy) < crosscov_tol

    Parameters
    ----------
    y : 1D array-like
        Observations
    f_fun : callable
        State transition function
    h_fun : callable
        Observation function
    sigma_eta : float
        State noise std
    sigma_e : float
        Observation noise std
    m0 : float
        Initial mean
    P0 : float
        Initial variance
    method : {"EKF","UKF"}
    return_diagnostics : bool
        If True, return filtering diagnostics
    stop_on_failure : bool
        If True, stop when a method-specific degeneracy is detected
    x_tol : float
        Threshold defining the bad-geometry region around zero
    crosscov_tol : float
        Threshold for UKF cross-covariance degeneracy
    jf_fun : callable or None
        Explicit derivative of f_fun, if available
    jh_fun : callable or None
        Explicit derivative of h_fun, if available
    """

    # ---------- Method check ----------
    method = method.upper()
    if method not in ("EKF", "UKF"):
        raise ValueError("method must be 'EKF' or 'UKF'")

    # ---------- Convert inputs ----------
    y = tf.convert_to_tensor(y, dtype=tf.float64)
    m0 = tf.convert_to_tensor(m0, dtype=tf.float64)
    P0 = tf.convert_to_tensor(P0, dtype=tf.float64)
    sigma_eta = tf.convert_to_tensor(sigma_eta, dtype=tf.float64)
    sigma_e = tf.convert_to_tensor(sigma_e, dtype=tf.float64)
    x_tol = tf.convert_to_tensor(x_tol, dtype=tf.float64)
    crosscov_tol = tf.convert_to_tensor(crosscov_tol, dtype=tf.float64)

    # ---------- Checks ----------
    if y.shape.ndims != 1 or tf.size(y) < 2:
        raise ValueError("y must be a 1D vector of length >= 2")
    if not tf.reduce_all(tf.math.is_finite(y)):
        raise ValueError("the values of y must be finite")

    for val, name in [(m0, "m0"), (P0, "P0"), (sigma_eta, "sigma_eta"), (sigma_e, "sigma_e")]:
        if not tf.reduce_all(tf.math.is_finite(val)):
            raise ValueError(f"{name} must be finite")
        if val.shape.ndims != 0:
            raise ValueError(f"{name} must be a scalar")

    if P0 <= 0 or sigma_eta <= 0 or sigma_e <= 0:
        raise ValueError("P0, sigma_eta, and sigma_e must be positive")

    if not callable(f_fun):
        raise ValueError("f_fun must be callable")
    if not callable(h_fun):
        raise ValueError("h_fun must be callable")

    n = tf.shape(y)[0]
    pi_tf = tf.constant(math.pi, dtype=tf.float64)

    # ---------- Allocation ----------
    mu_pred = tf.Variable(tf.zeros(n, dtype=tf.float64))
    mu_filt = tf.Variable(tf.zeros(n, dtype=tf.float64))
    P_pred  = tf.Variable(tf.zeros(n, dtype=tf.float64))
    P_filt  = tf.Variable(tf.zeros(n, dtype=tf.float64))
    v       = tf.Variable(tf.zeros(n, dtype=tf.float64))
    llk     = tf.Variable(0.0, dtype=tf.float64)

    # diagnostics
    Jh_vec = tf.Variable(tf.fill([n], tf.constant(np.nan, dtype=tf.float64)))   # EKF
    K_vec  = tf.Variable(tf.fill([n], tf.constant(np.nan, dtype=tf.float64)))
    S_vec  = tf.Variable(tf.fill([n], tf.constant(np.nan, dtype=tf.float64)))
    Cxy_vec = tf.Variable(tf.fill([n], tf.constant(np.nan, dtype=tf.float64)))  # UKF
    mu_y_vec = tf.Variable(tf.fill([n], tf.constant(np.nan, dtype=tf.float64))) # UKF

    # flags
    bad_geom_flag = tf.Variable(tf.zeros(n, dtype=tf.int32))   # |mu_pred| < x_tol
    bad_sigma_flag = tf.Variable(tf.zeros(n, dtype=tf.int32))  # UKF: |C_xy| small
    bad_flag = tf.Variable(tf.zeros(n, dtype=tf.int32))        # method-specific summary

    # ---------- Initialization ----------
    mu_filt[0].assign(m0)
    P_filt[0].assign(P0)
    mu_pred[0].assign(m0)
    P_pred[0].assign(P0)
    v[0].assign(tf.constant(0.0, dtype=tf.float64))

    t_indices = tf.range(1, n, dtype=tf.int32)

    # ---------- Recursion ----------
    for t in t_indices:
        # ----- Prediction -----
        if method == "EKF":
            if jf_fun is not None:
                Jf = jf_fun(mu_filt[t - 1])
            else:
                Jf = compute_jacobian_tf(f_fun, tf.identity(mu_filt[t - 1]), tol=0.0)

            mu_pred_t = f_fun(mu_filt[t - 1])
            P_pred_t = Jf**2 * P_filt[t - 1] + sigma_eta**2

        else:  # UKF
            sp = compute_sigma_points_tf(mu_filt[t - 1], P_filt[t - 1])
            X, Wm, Wc = sp["X"], sp["Wm"], sp["Wc"]

            X_pred = tf.stack([f_fun(xi) for xi in X])

            if not tf.reduce_all(tf.math.is_finite(X_pred)):
                raise ValueError(f"Non-finite prediction at t={t}")

            mu_pred_t = tf.reduce_sum(Wm * X_pred)
            P_pred_t = sigma_eta**2 + tf.reduce_sum(Wc * (X_pred - mu_pred_t)**2)

        if not tf.math.is_finite(P_pred_t) or P_pred_t <= 0:
            raise ValueError(f"Invalid P_pred at t={t}")

        mu_pred[t].assign(mu_pred_t)
        P_pred[t].assign(P_pred_t)

        # ----- Common bad geometry diagnostic -----
        geom_t = tf.cast(tf.abs(mu_pred_t) < x_tol, tf.int32)
        bad_geom_flag[t].assign(geom_t)

        # ----- Update -----
        if method == "EKF":
            if jh_fun is not None:
                Jh = jh_fun(mu_pred_t)
            else:
                Jh = compute_jacobian_tf(h_fun, tf.identity(mu_pred_t), tol=0.0)

            Jh_vec[t].assign(Jh)

            # For EKF, bad flag is the geometry flag
            bad_flag[t].assign(geom_t)

            if stop_on_failure and geom_t == 1:
                raise ValueError(f"EKF bad geometry detected at t={t}: |mu_pred| < x_tol")

            v_t = y[t] - h_fun(mu_pred_t)
            S_t = Jh**2 * P_pred_t + sigma_e**2

            if not tf.math.is_finite(S_t) or S_t <= 0:
                raise ValueError(f"Invalid S_t at t={t}")

            K = P_pred_t * Jh / S_t
            mu_t_filt = mu_pred_t + K * v_t
            P_t_filt = (1.0 - K * Jh) * P_pred_t

        else:  # UKF
            sp = compute_sigma_points_tf(mu_pred_t, P_pred_t)
            X, Wm, Wc = sp["X"], sp["Wm"], sp["Wc"]

            Y = tf.stack([h_fun(xi) for xi in X])
            if not tf.reduce_all(tf.math.is_finite(Y)):
                raise ValueError(f"Non-finite observation mapping at t={t}")

            mu_y = tf.reduce_sum(Wm * Y)
            mu_y_vec[t].assign(mu_y)

            S_t = sigma_e**2 + tf.reduce_sum(Wc * (Y - mu_y)**2)
            if not tf.math.is_finite(S_t) or S_t <= 0:
                raise ValueError(f"Invalid S_t at t={t}")

            C_xy = tf.reduce_sum(Wc * (X - mu_pred_t) * (Y - mu_y))
            Cxy_vec[t].assign(C_xy)

            sigma_t = tf.cast(tf.abs(C_xy) < crosscov_tol, tf.int32)
            bad_sigma_flag[t].assign(sigma_t)

            # For UKF, define bad flag as union of bad geometry and sigma-point degeneracy
            bad_t = tf.cast(tf.logical_or(tf.cast(geom_t, tf.bool), tf.cast(sigma_t, tf.bool)), tf.int32)
            bad_flag[t].assign(bad_t)

            if stop_on_failure and sigma_t == 1:
                raise ValueError(f"UKF sigma-point degeneracy detected at t={t}: |C_xy| < crosscov_tol")

            K = C_xy / S_t
            v_t = y[t] - mu_y
            mu_t_filt = mu_pred_t + K * v_t
            P_t_filt = P_pred_t - K**2 * S_t

        # ----- State checks -----
        if not tf.math.is_finite(mu_t_filt) or not tf.math.is_finite(P_t_filt):
            raise ValueError(f"Invalid filtered state at t={t}")

        P_t_filt = tf.maximum(P_t_filt, tf.constant(1e-12, dtype=tf.float64))

        mu_filt[t].assign(mu_t_filt)
        P_filt[t].assign(P_t_filt)
        v[t].assign(v_t)
        K_vec[t].assign(K)
        S_vec[t].assign(S_t)

        # ----- Likelihood -----
        llk_inc = -0.5 * (tf.math.log(2.0 * pi_tf * S_t) + v_t**2 / S_t)
        if not tf.math.is_finite(llk_inc):
            raise ValueError(f"Invalid log-likelihood increment at t={t}")

        llk.assign_add(llk_inc)

    out = {
        "mu_filt": mu_filt,
        "P_filt": P_filt,
        "mu_pred": mu_pred,
        "P_pred": P_pred,
        "llk": llk
    }

    if return_diagnostics:
        out["diagnostics"] = {
            "Jh": Jh_vec,
            "K": K_vec,
            "S": S_vec,
            "v": v,
            "C_xy": Cxy_vec,
            "mu_y": mu_y_vec,
            "bad_geom_flag": bad_geom_flag,
            "bad_sigma_flag": bad_sigma_flag,
            "bad_flag": bad_flag,
            "frac_bad_geom": tf.reduce_mean(tf.cast(bad_geom_flag, tf.float64)),
            "frac_bad_sigma": tf.reduce_mean(tf.cast(bad_sigma_flag, tf.float64)),
            "frac_bad": tf.reduce_mean(tf.cast(bad_flag, tf.float64)),
            "n_bad_geom": tf.reduce_sum(bad_geom_flag),
            "n_bad_sigma": tf.reduce_sum(bad_sigma_flag),
            "n_bad": tf.reduce_sum(bad_flag)
        }

    return out

#### Examples

In [ ]:
#### SIMULATE MODEL 
def simulate_quadratic_model(T, phi, sigma_eta, sigma_e, dtype=tf.float64, seed=42):
    tf.random.set_seed(seed)
    # float64 is numerically safer for EKF/UKF diagnostics

    x0 = tf.random.normal([], mean=0.0, stddev=1.0, dtype=dtype)
    x = tf.TensorArray(tf.float64, size=T)
    x = x.write(0, x0)

    for t in range(1, T):
        eta_t = tf.random.normal([], mean=0.0, stddev=sigma_eta, dtype=dtype)
        x_t = phi * x.read(t - 1) + eta_t
        x = x.write(t, x_t)

    x = x.stack()
    y = x**2 / 5.0 + tf.random.normal([T], mean=0.0, stddev=sigma_e, dtype=dtype)

    return {"x": x, "y": y}


def make_quadratic_model_components(phi):
    def f_fun_tf(x):
        return phi * x

    def h_fun_tf(x):
        return x**2 / 5.0

    def jf_fun_tf(x):
        return tf.constant(phi, dtype=tf.float64)

    def jh_fun_tf(x):
        return 2.0 * x / 5.0

    return {
        "f_fun": f_fun_tf,
        "h_fun": h_fun_tf,
        "jf_fun": jf_fun_tf,
        "jh_fun": jh_fun_tf
    }


#### Function to RUN EXPERIMENTS

def run_heatmap_analysis(
    simulator_fun,
    phi_grid,
    sigma_e_grid,
    sigma_eta=0.2,
    T=100,
    m0=0.0,
    P0=1.0,
    x_tol=0.05,
    crosscov_tol=1e-10,
    seed=42
):
    rows = []
    diagnostics_store = {}

    for phi in phi_grid:
        components = make_quadratic_model_components(phi)

        for sigma_e in sigma_e_grid:
            sim_out = simulator_fun(T=T, phi=phi, sigma_eta=sigma_eta, sigma_e=sigma_e, seed=seed)

            x = sim_out["x"]
            y = sim_out["y"]

            res_ekf = EKF_UKF_univariate(
                y=y,
                f_fun=components["f_fun"],
                h_fun=components["h_fun"],
                sigma_eta=sigma_eta,
                sigma_e=sigma_e,
                m0=tf.constant(m0, dtype=tf.float64),
                P0=tf.constant(P0, dtype=tf.float64),
                method="EKF",
                return_diagnostics=True,
                stop_on_failure=False,
                x_tol=x_tol,
                jf_fun=components["jf_fun"],
                jh_fun=components["jh_fun"]
            )

            res_ukf = EKF_UKF_univariate(
                y=y,
                f_fun=components["f_fun"],
                h_fun=components["h_fun"],
                sigma_eta=sigma_eta,
                sigma_e=sigma_e,
                m0=tf.constant(m0, dtype=tf.float64),
                P0=tf.constant(P0, dtype=tf.float64),
                method="UKF",
                return_diagnostics=True,
                stop_on_failure=False,
                x_tol=x_tol,
                crosscov_tol=crosscov_tol
            )

            rmse_ekf = tf.sqrt(tf.reduce_mean((res_ekf["mu_filt"] - x)**2)).numpy()
            rmse_ukf = tf.sqrt(tf.reduce_mean((res_ukf["mu_filt"] - x)**2)).numpy()

            rows.append({
                "phi": phi,
                "sigma_e": sigma_e,
                "ekf_frac_bad_geom": float(res_ekf["diagnostics"]["frac_bad_geom"].numpy()),
                "ukf_frac_bad_geom": float(res_ukf["diagnostics"]["frac_bad_geom"].numpy()),
                "ukf_frac_bad_sigma": float(res_ukf["diagnostics"]["frac_bad_sigma"].numpy()),
                "ukf_frac_bad_total": float(res_ukf["diagnostics"]["frac_bad"].numpy()),
                "ekf_rmse": float(rmse_ekf),
                "ukf_rmse": float(rmse_ukf),
                "ekf_loglik": float(res_ekf["llk"].numpy()),
                "ukf_loglik": float(res_ukf["llk"].numpy())
            })

            diagnostics_store[(phi, sigma_e)] = {
                "x": x,
                "y": y,
                "ekf": res_ekf,
                "ukf": res_ukf
            }

    df = pd.DataFrame(rows)
    return df, diagnostics_store

def make_color_cmap():
    return LinearSegmentedColormap.from_list(
        "green_orange_red",
        ["#2E8B57", "#F4A261", "#D62828"]
    )


def plot_heatmaps_quadratic(df):
    cmap = make_color_cmap()
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    # Precompute pivots once
    piv = lambda col: df.pivot(index="phi", columns="sigma_e", values=col)

    heatmaps = [
        ("ekf_frac_bad_geom", "EKF: fraction in bad geometry", ".2f", cmap, (0, 1)),
        ("ukf_frac_bad_sigma", "UKF: sigma-point degeneracy", ".2f", cmap, (0, 1)),
        ("ukf_frac_bad_total", "UKF: total bad fraction", ".2f", cmap, (0, 1)),
        ("ekf_rmse", "EKF RMSE", ".3f", cmap, None),
        ("ukf_rmse", "UKF RMSE", ".3f", cmap, None),
    ]

    # Plot first 5 heatmaps
    for ax, (col, title, fmt, cm, lims) in zip(axes.flat[:5], heatmaps):
        data = piv(col)
        sns.heatmap(
            data,
            annot=True,
            fmt=fmt,
            cmap=cm,
            vmin=lims[0] if lims else None,
            vmax=lims[1] if lims else None,
            ax=ax
        )
        ax.set_title(title)
        ax.set_xlabel("sigma_eps")
        ax.set_ylabel("phi")

    # RMSE difference
    heat_diff = piv("ukf_rmse") - piv("ekf_rmse")
    sns.heatmap(heat_diff, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=axes.flat[5])

    axes.flat[5].set_title("UKF RMSE - EKF RMSE")
    axes.flat[5].set_xlabel("sigma_eps")
    axes.flat[5].set_ylabel("phi")

    plt.tight_layout()
    plt.show()


def print_summary_stats(df):
    cols = [
        "ekf_frac_bad_geom",
        "ukf_frac_bad_geom",
        "ukf_frac_bad_sigma",
        "ukf_frac_bad_total",
        "ekf_rmse",
        "ukf_rmse",
        "ekf_loglik",
        "ukf_loglik"
    ]
    print(df[cols].describe().round(4))


def plot_fraction_lollipops(df):
    metrics = [
        ("ekf_frac_bad_geom", "EKF bad geometry", "#2E8B57"),
        ("ukf_frac_bad_sigma", "UKF sigma-point degeneracy", "#F4A261"),
        ("ukf_frac_bad_total", "UKF total bad fraction", "#D62828"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharex=False)

    for ax, (col, title, color) in zip(axes, metrics):
        df_plot = df.copy()
        df_plot["config"] = df_plot.apply(
            lambda r: f"(phi={r['phi']}, sigma_eps={r['sigma_e']})", axis=1
        )
        df_plot = df_plot.sort_values(col).reset_index(drop=True)

        y = np.arange(len(df_plot))
        x = df_plot[col].to_numpy()

        ax.hlines(y=y, xmin=0, xmax=x, color=color, alpha=0.75, linewidth=2)
        ax.scatter(x, y, color=color, s=50, zorder=3)

        ax.set_yticks(y)
        ax.set_yticklabels(df_plot["config"])
        ax.set_xlim(0, 1.02)
        ax.set_xlabel("Fraction")
        ax.set_title(title)
        ax.grid(axis="x", alpha=0.25)

        ax.axvline(np.mean(x), color="black", linestyle="--", linewidth=1.2, label="mean")
        ax.axvline(np.median(x), color="black", linestyle=":", linewidth=1.2, label="median")
        ax.legend()

    plt.tight_layout()
    plt.show()


# ACF and PACF analysis

def find_bad_pair(diagnostics_store, phi_target, sigma_e_target, tol=1e-10):
    for key in diagnostics_store.keys():
        phi_k, sigma_e_k = key
        if abs(float(phi_k) - float(phi_target)) < tol and abs(float(sigma_e_k) - float(sigma_e_target)) < tol:
            return key
    raise KeyError(
        f"No bad pair found for (phi={phi_target}, sigma_e={sigma_e_target}). "
        f"Available pairs are: {list(diagnostics_store.keys())}"
    )



def plot_bad_acf(diagnostics_store, selected_configs, nlags=30):
    valid_keys = list(diagnostics_store.keys())

    print("\nAvailable parameter combinations:")
    for k in valid_keys:
        print(k)

    matched_keys = []
    for cfg in selected_configs:
        if cfg in diagnostics_store:
            matched_keys.append(cfg)
        else:
            print(f"\nSkipping {cfg}: not available in diagnostics_store")

    if len(matched_keys) == 0:
        raise ValueError("None of the selected configurations are available.")

    print("\nConfigurations used in the ACF plot:")
    for k in matched_keys:
        print(k)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    ax_ekf = axes[0]
    ax_ukf = axes[1]

    conf_ekf_ref = None
    conf_ukf_ref = None

    for key in matched_keys:
        phi, sigma_e = key
        out = diagnostics_store[key]

        ekf_bad = out["ekf"]["diagnostics"]["bad_geom_flag"].numpy().astype(float)
        ukf_bad = out["ukf"]["diagnostics"]["bad_sigma_flag"].numpy().astype(float)

        ekf_n_bad = int(np.sum(ekf_bad))
        ukf_n_bad = int(np.sum(ukf_bad))

        print(f"\nConfiguration phi={phi}, sigma_e={sigma_e}")
        print(f"EKF: number of bad-geometry time points = {ekf_n_bad}")
        print(f"EKF: fraction in bad geometry = {np.mean(ekf_bad):.3f}")
        print(f"UKF: number of sigma-point degeneracy time points = {ukf_n_bad}")
        print(f"UKF: fraction in sigma-point degeneracy = {np.mean(ukf_bad):.3f}")

        ekf_acf = acf(ekf_bad, nlags=nlags, fft=True)
        ukf_acf = acf(ukf_bad, nlags=nlags, fft=True)

        lags = np.arange(len(ekf_acf))

        conf_ekf_ref = 1.96 / np.sqrt(len(ekf_bad))
        conf_ukf_ref = 1.96 / np.sqrt(len(ukf_bad))

        ax_ekf.vlines(lags, 0, ekf_acf, linewidth=2, alpha=0.9, label=f"phi={phi}, σe={sigma_e}")
        ax_ekf.plot(lags, ekf_acf, "o", markersize=3)

        ax_ukf.vlines(lags, 0, ukf_acf, linewidth=2, alpha=0.9, label=f"phi={phi}, σe={sigma_e}")
        ax_ukf.plot(lags, ukf_acf, "o", markersize=3)

    ax_ekf.axhline(0, color="black", linewidth=1)
    ax_ukf.axhline(0, color="black", linewidth=1)

    if conf_ekf_ref is not None:
        ax_ekf.axhline(conf_ekf_ref, color="black", linestyle="--", linewidth=1)
        ax_ekf.axhline(-conf_ekf_ref, color="black", linestyle="--", linewidth=1)

    if conf_ukf_ref is not None:
        ax_ukf.axhline(conf_ukf_ref, color="black", linestyle="--", linewidth=1)
        ax_ukf.axhline(-conf_ukf_ref, color="black", linestyle="--", linewidth=1)

    ax_ekf.set_title("ACF of EKF bad-geometry indicator")
    ax_ukf.set_title("ACF of UKF sigma-point degeneracy indicator")

    for ax in axes:
        ax.set_xlabel("Lag")
        ax.set_ylabel("ACF")
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

    fig.suptitle(
        "Selected configurations: " +
        ", ".join([f"(phi={phi}, sigma_e={sigma_e})" for phi, sigma_e in matched_keys]),
        fontsize=10
    )

    plt.tight_layout()
    plt.show()



def plot_bad_pacf(
    diagnostics_store,
    selected_configs,
    nlags=30,
    save=False,
    save_dir="figures",
    filename="bad_pacf.png",
    dpi=300,
    close_after_save=False,
):
    valid_keys = list(diagnostics_store.keys())

    print("\nAvailable parameter combinations:")
    for k in valid_keys:
        print(k)

    matched_keys = []
    for cfg in selected_configs:
        if cfg in diagnostics_store:
            matched_keys.append(cfg)
        else:
            print(f"\nSkipping {cfg}: not available in diagnostics_store")

    if len(matched_keys) == 0:
        raise ValueError("None of the selected configurations are available.")

    print("\nConfigurations used in the PACF plot:")
    for k in matched_keys:
        print(k)

    # bright colors for dots
    colors = sns.color_palette("bright", len(matched_keys))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax_ekf, ax_ukf = axes

    # neutral bar color
    bar_color = "#B0BEC5"   # light blue-grey

    conf = None  # will set from sample size below

    for idx, key in enumerate(matched_keys):
        phi, sigma_e = key
        color = colors[idx]

        out = diagnostics_store[key]

        ekf_bad = out["ekf"]["diagnostics"]["bad_geom_flag"].numpy().astype(float)
        ukf_bad = out["ukf"]["diagnostics"]["bad_sigma_flag"].numpy().astype(float)

        ekf_n_bad = int(np.sum(ekf_bad))
        ukf_n_bad = int(np.sum(ukf_bad))

        print(f"\nConfiguration phi={phi}, sigma_e={sigma_e}")
        print(f"EKF: number of bad-geometry time points = {ekf_n_bad}")
        print(f"EKF: fraction in bad geometry = {np.mean(ekf_bad):.3f}")
        print(f"UKF: number of sigma-point degeneracy time points = {ukf_n_bad}")
        print(f"UKF: fraction in sigma-point degeneracy = {np.mean(ukf_bad):.3f}")

        # PACF
        ekf_pacf = pacf(ekf_bad, nlags=nlags, method="ywm")
        ukf_pacf = pacf(ukf_bad, nlags=nlags, method="ywm")

        lags = np.arange(len(ekf_pacf))

        conf = 1.96 / np.sqrt(len(ekf_bad))

        # --- EKF ---
        ax_ekf.vlines(
            lags, 0, ekf_pacf,
            color=bar_color,
            linewidth=2,
            alpha=0.6,
            zorder=1
        )

        ax_ekf.scatter(
            lags, ekf_pacf,
            s=30,
            color=color,
            label=f"(φ={phi}, σe={sigma_e})",
            zorder=3
        )

        # --- UKF ---
        ax_ukf.vlines(
            lags, 0, ukf_pacf,
            color=bar_color,
            linewidth=2,
            alpha=0.6,
            zorder=1
        )

        ax_ukf.scatter(
            lags, ukf_pacf,
            s=30,
            color=color,
            label=f"(φ={phi}, σe={sigma_e})",
            zorder=3
        )

    # zero line
    ax_ekf.axhline(0, color="black", linewidth=1)
    ax_ukf.axhline(0, color="black", linewidth=1)

    # CI
    if conf is not None:
        ax_ekf.axhline(conf, color="black", linestyle="--", linewidth=1.2)
        ax_ekf.axhline(-conf, color="black", linestyle="--", linewidth=1.2)

        ax_ukf.axhline(conf, color="black", linestyle="--", linewidth=1.2)
        ax_ukf.axhline(-conf, color="black", linestyle="--", linewidth=1.2)

    # titles
    ax_ekf.set_title("PACF of the EKF bad-geometry indicator")
    ax_ukf.set_title("PACF of UKF sigma-point degeneracy indicator")

    for ax in axes:
        ax.set_xlabel("Lag")
        ax.set_ylabel("PACF Value")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=9)

    fig.suptitle("PACF across parameter configurations", fontsize=11)

    plt.tight_layout()

    if save:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, filename)
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"\nSaved figure to: {save_path}")

    plt.show()

    if close_after_save:
        plt.close(fig)

    return fig, axes


#### Run Experiments 

In [ ]:
phi_grid = [0.3, 0.5, 0.9, 0.95]
sigma_e_grid = [0.05, 0.50, 1.0, 2.0]

df_heat, diagnostics_store = run_heatmap_analysis(
    simulator_fun=simulate_quadratic_model,
    phi_grid=phi_grid,
    sigma_e_grid=sigma_e_grid,
    sigma_eta=0.2,
    T=70,
    m0=0.5,
    P0=1.0,
    x_tol=0.05,
    crosscov_tol=1e-10,
    seed=42
)

plot_heatmaps_quadratic(df_heat)

print_summary_stats(df_heat)
plot_fraction_lollipops(df_heat)
plot_heatmaps_quadratic(df_heat)


selected = [(0.3, 0.05), (0.5, 0.05), (0.95, 0.05), (0.95, 0.50), (0.95, 2.0)]
plot_bad_acf(diagnostics_store, selected_configs=selected, nlags=25)
plot_bad_pacf(diagnostics_store, selected_configs=selected, nlags=25)